# Fiabilisation de la base Diplômes

Ce notebook reprend, étape par étape, la fiabilisation de l'extraction *BDD Diplômes*. Chaque étape suit la même structure :

1. **Objectif** : ce que l'étape doit régler ;
2. **Choix** : la règle retenue et pourquoi ;
3. **Code** : la transformation ;
4. **Contrôle** : un tableau avant/après sur les données réelles.

| Étape | Contenu |
|---|---|
| 0 | Chargement et diagnostic |
| 1 | Nettoyage technique (espaces, casse, accents, vides déguisés) |
| 2 | Normalisation du niveau saisi |
| 3 | Classification des libellés de diplôme |
| 4 | Niveau fiabilisé (complétion, incohérences, arrondi des « et plus ») |
| 5 | École (décodage des codes SIRH, pays, nature, confiance) et catégorie du diplôme |
| 6 | Dédoublonnage |
| 7 | Contrôles et statut par ligne |
| 8 | Vue salarié |
| 9 | Export et synthèse |

**Principe général** : la colonne source est toujours gardée. Quand une étape produit une nouvelle version d'une colonne, elle remplace la version intermédiaire ; les colonnes de contrôle (`ctrl_…`) sont gardées. Chaque étape se termine par un **aperçu de la base** pour voir ce qu'elle devient.

**Pour l'ouvrir** : double-cliquer sur `lancer_notebook.bat`, dans le même dossier que ce notebook. Il s'ouvre dans le navigateur, déjà branché sur le Python du projet (« Python (.venv salaires) ») : rien à installer, rien à choisir.

**Après une modification** : relancer depuis le début (menu *Run > Run All Above Selected Cell*), car les colonnes intermédiaires n'existent plus une fois remplacées.

**Pour relancer sur une nouvelle extraction** : déposer le fichier Excel dans ce dossier, à côté du notebook, changer `FICHIER_SOURCE` dans la cellule suivante, puis *Run > Run All Cells*. Le fichier de sortie (`FICHIER_SORTIE`) est écrit dans ce même dossier.

In [1]:
import re
import unicodedata

import numpy as np
import pandas as pd

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", None)   # aperçus : toutes les colonnes visibles

# Seules lignes à changer pour une nouvelle extraction
# (le fichier source est lu, et le fichier de sortie écrit, dans le dossier de ce notebook)
FICHIER_SOURCE = "Diplômes à fiabiliser - CB 19102026.xlsx"
FICHIER_SORTIE = "BDD_Diplomes_FINAL.xlsx"

---
## Étape 0 : Chargement et diagnostic

**Objectif** : mesurer l'état du fichier avant toute modification, pour pouvoir comparer à la fin.

Les colonnes sont renommées avec des noms courts pour la lisibilité du code. Un numéro de ligne Excel d'origine (`ligne_source`) est ajouté pour pouvoir toujours revenir à la ligne du fichier initial.

In [2]:
# Noms courts pour la lisibilité du code
COLONNES = {
    "Matricule fictif": "matricule",
    "Nom de diplôme ou certification": "diplome",
    "Niveau du diplôme": "niveau",
    "Spécialité du diplôme": "specialite",
    "Ecole Diplome ou Organisme Certification": "ecole",
    "Discipline": "discipline",
}

brut = pd.read_excel(FICHIER_SOURCE)
# Arrêt immédiat si l'extraction n'a pas les colonnes attendues : mieux vaut une erreur qu'un résultat faux
manquantes = [c for c in COLONNES if c not in brut.columns]
assert not manquantes, f"Colonnes absentes du fichier : {manquantes}"

df = brut.rename(columns=COLONNES)[list(COLONNES.values())].copy()
df["ligne_source"] = df.index + 2          # numéro de ligne Excel d'origine (en-tête = ligne 1), pour retrouver la ligne source

COLS = list(COLONNES.values())
print(f"{len(df):,} lignes  |  {df['matricule'].nunique():,} matricules".replace(",", " "))

27 242 lignes  |  8 715 matricules


In [3]:
# Remplissage et diversité de chaque colonne
pd.DataFrame({
    "valeurs_distinctes": df[COLS].nunique(),
    "lignes_vides": df[COLS].isna().sum(),
    "% vides": (df[COLS].isna().mean() * 100).round(1),
})

,valeurs_distinctes,lignes_vides,% vides
matricule,8715,0,0.0
diplome,214,0,0.0
niveau,25,11794,43.3
specialite,83,1031,3.8
ecole,334,1149,4.2
discipline,28,24265,89.1


In [4]:
print("Doublons strictement identiques (avant tout nettoyage) :", df.duplicated(subset=COLS).sum())

print("X salariés ont N lignes dans le fichier")
# Nombre de lignes par salarié
(df.groupby("matricule").size().value_counts().sort_index()
   .rename_axis("lignes_par_salarie").to_frame("nb_salaries").T)


Doublons strictement identiques (avant tout nettoyage) : 6389
X salariés ont N lignes dans le fichier


lignes_par_salarie,1,2,3,4,5,6,7,8,9,10,11,12,14,17,18,24,26,63
nb_salaries,865,3439,1366,1858,168,648,24,264,13,43,2,18,2,1,1,1,1,1


In [5]:
# Les écritures du niveau telles qu'elles existent dans la source
df["niveau"].value_counts(dropna=False).to_frame("nb_lignes")

,nb_lignes
niveau,
NaN,11794
BAC+2,2398
Bac + 5,2182
NIVEAU SECONDAIRE,2079
Bac + 3,1841
BAC+5,1685
BAC,1354
BAC+4,1220
Bac,906


**Ce qu'on retient du diagnostic**

- Près d'un quart des lignes sont des doublons stricts : l'extraction concatène probablement plusieurs exports.
- Le niveau est vide sur plus de 40 % des lignes et s'écrit de 25 façons différentes.
- La colonne *Discipline* n'est remplie que sur 11 % des lignes, tous diplômes confondus.

In [6]:
# Aperçu de la base après l'étape 0
df.head(10)

,matricule,diplome,niveau,specialite,ecole,discipline,ligne_source
0,MATRICULE000001,SECONDAIRE,SECONDAIRE,Autres,LYCEE,NaN,2
1,MATRICULE000002,BREVET TECHNICIEN SPECIALISE,BAC+2,Autres,AUTRES,NaN,3
2,MATRICULE000002,BACCALAUREAT,BAC,Autres,LYCEE,SCIENCE,4
3,MATRICULE000002,CERTIFICAT D'APTITUDE BANCAIRE,BAC,DOMAINE BANCAIRE,C.F.P.B.,NaN,5
4,MATRICULE000003,DIPLOME DES ETUDES UNIVERSITAIRES GNLE.,BAC+2,DROIT,UNIVMARCDR,DROIT,6
5,MATRICULE000003,DIPLOME DES ETUDES UNIVERSITAIRES GNLE.,BAC+2,DROIT,UNIVMARCDR,JURIDIQUE,7
6,MATRICULE000003,BACCALAUREAT,BAC,LETTRES MODERNES,LYCEE,LETTRES MODERNES,8
7,MATRICULE000004,SECONDAIRE,SECONDAIRE,Autres,LYCEE,NaN,9
8,MATRICULE000005,INSTITUT DES TECHNIQUES DE BANQUE,BAC+4,DOMAINE BANCAIRE,C.F.P.B.,NaN,10
9,MATRICULE000005,BREVET BANCAIRE,BAC+2,DOMAINE BANCAIRE,C.F.P.B.,NaN,11


---
## Étape 1 : Nettoyage technique

**Objectif** : qu'une même information soit toujours écrite de la même façon, pour que les étapes suivantes (classement, dédoublonnage) ne soient pas trompées par la forme.

**Règles appliquées** aux colonnes Diplôme, Niveau, École, Spécialité et Discipline (ces deux dernières servent uniquement à reconnaître les doublons à l'étape 6). Le matricule n'est pas touché : c'est un identifiant, déjà propre.

- suppression des espaces en début/fin et des doubles espaces ;
- passage en MAJUSCULES et suppression des accents (`Diplôme d'Ingénieur` → `DIPLOME D'INGENIEUR`) ;
- espaces normalisés autour de `/ & , :` (`A/B` et `A / B` deviennent identiques) ;
- pour le niveau, aucun espace autour du `+` (`Bac + 3`, `Bac +3` → `BAC+3`) et « ou » écrit en entier (`Bac + 3 ou 4` → `BAC+3 OU BAC+4`) ;
- les **vides déguisés** deviennent de vrais vides : `-`, `AUTRES`, `DEFAUT`, `N/A`, `.`, `?`…

**Choix — `AUTRES` et `DEFAUT` = non renseigné.** On pourrait les garder comme catégorie. Ils sont écartés parce que :

- « Autres » ne dit rien sur la spécialité ou l'école réelle ;
- comme catégorie, ce serait l'une des plus fréquentes du fichier et elle fausserait tout comptage ou modèle ;
- un vide explicite est visible dans les contrôles et peut être complété à la source, alors qu'« Autres » passe inaperçu.


In [7]:
# Vides déguisés : valeurs qui ne disent rien, transformées en vrais vides.
# AUTRE(S) et DEFAUT en font partie : ils ne désignent aucune école ni spécialité réelle,
# et un vide se voit dans les contrôles alors qu'« Autres » passe inaperçu.
PLACEHOLDERS = {"", "-", "--", ".", "?", "NA", "N/A", "NAN", "NONE", "NULL",
                "AUTRE", "AUTRES", "DEFAUT", "INCONNU", "NON RENSEIGNE", "A COMPLETER"}

def normaliser_texte(x):
    """Majuscules, sans accents, espaces simples ; vides déguisés -> None."""
    if pd.isna(x):
        return None
    x = re.sub(r"[‘’ʼ´`]", "'", str(x))   # apostrophe typographique -> ' (sinon elle serait supprimée à la ligne suivante)
    s = unicodedata.normalize("NFKD", x).encode("ascii", "ignore").decode()   # sans accents : « Diplôme » = « DIPLOME »
    s = re.sub(r"\s+", " ", s).strip().upper()
    s = re.sub(r"\s*([/&,:])\s*", r" \1 ", s)                 # « A/B » et « A / B » s'écrivent pareil
    s = re.sub(r"\s+", " ", s).strip()
    return None if s in PLACEHOLDERS else s

# Spécialité et discipline : nettoyées seulement pour reconnaître les doublons (étape 6)
# Matricule : pas nettoyé, c'est un identifiant déjà propre
COLS_TEXTE = ["diplome", "niveau", "specialite", "ecole", "discipline"]
for c in COLS_TEXTE:
    df[f"{c}_norm"] = df[c].map(normaliser_texte)

# Niveau : une seule façon d'écrire (« Bac + 3 », « Bac +3 » -> « BAC+3 » ; « Bac + 3 ou 4 » -> « BAC+3 OU BAC+4 »)
df["niveau_norm"] = (df["niveau_norm"]
                     .str.replace(r"\s*\+\s*", "+", regex=True)
                     .str.replace(r"^BAC\+(\d) OU (\d)$", r"BAC+\1 OU BAC+\2", regex=True))

In [8]:
# Contrôle : diversité et vides avant / après
pd.DataFrame({c: {
    "distinct_avant": df[c].nunique(),
    "distinct_apres": df[f"{c}_norm"].nunique(),
    "vides_avant": df[c].isna().sum(),
    "vides_apres": df[f"{c}_norm"].isna().sum(),
} for c in COLS_TEXTE}).T

,distinct_avant,distinct_apres,vides_avant,vides_apres
diplome,214,201,0,1
niveau,25,18,11794,11804
specialite,83,80,1031,2470
ecole,334,319,1149,2822
discipline,28,25,24265,24419


In [9]:
# Contrôle : quels vides déguisés ont été retirés, et combien
for c in COLS_TEXTE :
    s = df[c].astype("string").str.strip().str.upper()
    print(f"{c:<11}", s[s.isin(PLACEHOLDERS)].value_counts().to_dict())

diplome     {'-': 1}
niveau      {'-': 10}
specialite  {'AUTRES': 1386, 'DEFAUT': 27, '-': 26}
ecole       {'AUTRES': 1588, '-': 85}
discipline  {'DEFAUT': 137, '-': 17}


In [10]:
# Contrôle : exemples d'écritures différentes fusionnées par le nettoyage
def fusions(col, n=8):
    g = (df[[col, f"{col}_norm"]].dropna().drop_duplicates()
           .groupby(f"{col}_norm")[col].agg(list))
    return g[g.map(len) > 1].head(n).to_frame("ecritures_source_fusionnees")

display(fusions("niveau"))
display(fusions("diplome"))
display(fusions("ecole"))

,ecritures_source_fusionnees
niveau_norm,
BAC,"[BAC, Bac]"
BAC+1,"[Bac + 1, Bac +1]"
BAC+2,"[BAC+2, Bac + 2]"
BAC+3,"[Bac +3, Bac + 3]"
BAC+4,"[BAC+4, Bac + 4]"
BAC+5,"[BAC+5, Bac + 5]"


,ecritures_source_fusionnees
diplome_norm,
BREVET BANCAIRE,"[BREVET BANCAIRE, Brevet Bancaire]"
BREVET DE TECHNICIEN SUPERIEUR,"[BREVET DE TECHNICIEN SUPERIEUR, Brevet de technicien supérieur]"
CERTIFICAT D'APTITUDE PROFESSIONNELLE,"[CERTIFICAT D'APTITUDE PROFESSIONNELLE, Certificat d'aptitude professionnelle]"
DIPLOME,"[DIPLOME, Diplômé]"
DIPLOME D'INGENIEUR,"[Diplôme d'ingénieur, Diplôme d'Ingénieur]"
DIPLOME UNIVERSITAIRE DE TECHNOLOGIE,"[DIPLOME UNIVERSITAIRE DE TECHNOLOGIE, Diplôme universitaire de technologie]"
DOCTORAT,"[DOCTORAT, Doctorat]"
INGENIEUR,"[INGENIEUR, Ingénieur, Ingenieur]"


,ecritures_source_fusionnees
ecole_norm,
ATTIJARIWAFA BANK,"[Attijariwafa Bank, attijariwafa bank]"
ECP,"[ECP, ECP]"
ENIM,"[ENIM, ENIM]"
GOOGLE,"[Google, GOOGLE]"
GREAT LEARNING ACADEMY,"[Great LEARNING ACADEMY, Great learning academy]"
HARVARD BUSINESS PUBLISHING,"[Harvard Business Publishing, Harvard Business publishing]"
HEC PARIS,"[HEC Paris, HEC PARIS]"
IBM,"[IBM, ibm]"


In [11]:
# Aperçu de la base après l'étape 1
df.head(10)

,matricule,diplome,niveau,specialite,ecole,discipline,ligne_source,diplome_norm,niveau_norm,specialite_norm,ecole_norm,discipline_norm
0,MATRICULE000001,SECONDAIRE,SECONDAIRE,Autres,LYCEE,NaN,2,SECONDAIRE,SECONDAIRE,None,LYCEE,None
1,MATRICULE000002,BREVET TECHNICIEN SPECIALISE,BAC+2,Autres,AUTRES,NaN,3,BREVET TECHNICIEN SPECIALISE,BAC+2,None,None,None
2,MATRICULE000002,BACCALAUREAT,BAC,Autres,LYCEE,SCIENCE,4,BACCALAUREAT,BAC,None,LYCEE,SCIENCE
3,MATRICULE000002,CERTIFICAT D'APTITUDE BANCAIRE,BAC,DOMAINE BANCAIRE,C.F.P.B.,NaN,5,CERTIFICAT D'APTITUDE BANCAIRE,BAC,DOMAINE BANCAIRE,C.F.P.B.,None
4,MATRICULE000003,DIPLOME DES ETUDES UNIVERSITAIRES GNLE.,BAC+2,DROIT,UNIVMARCDR,DROIT,6,DIPLOME DES ETUDES UNIVERSITAIRES GNLE.,BAC+2,DROIT,UNIVMARCDR,DROIT
5,MATRICULE000003,DIPLOME DES ETUDES UNIVERSITAIRES GNLE.,BAC+2,DROIT,UNIVMARCDR,JURIDIQUE,7,DIPLOME DES ETUDES UNIVERSITAIRES GNLE.,BAC+2,DROIT,UNIVMARCDR,JURIDIQUE
6,MATRICULE000003,BACCALAUREAT,BAC,LETTRES MODERNES,LYCEE,LETTRES MODERNES,8,BACCALAUREAT,BAC,LETTRES MODERNES,LYCEE,LETTRES MODERNES
7,MATRICULE000004,SECONDAIRE,SECONDAIRE,Autres,LYCEE,NaN,9,SECONDAIRE,SECONDAIRE,None,LYCEE,None
8,MATRICULE000005,INSTITUT DES TECHNIQUES DE BANQUE,BAC+4,DOMAINE BANCAIRE,C.F.P.B.,NaN,10,INSTITUT DES TECHNIQUES DE BANQUE,BAC+4,DOMAINE BANCAIRE,C.F.P.B.,None
9,MATRICULE000005,BREVET BANCAIRE,BAC+2,DOMAINE BANCAIRE,C.F.P.B.,NaN,11,BREVET BANCAIRE,BAC+2,DOMAINE BANCAIRE,C.F.P.B.,None


---
## Étape 2 : Normalisation du niveau saisi

**Objectif** : ramener chaque niveau écrit (après l'étape 1) à un niveau standard, avec un rang pour comparer et trier.
Rangs : 0 SECONDAIRE · 1 BAC · 2 BAC+1 · 3 BAC+2 · 4 BAC+3 · 5 BAC+4 · 6 BAC+5 (et expertise comptable) · 7 BAC+6 (et plus) · 8 BAC+7 (doctorat).

| Niveau écrit | Devient | Rang | Ce qu'on en fait |
|---|---|---|---|
| `SECONDAIRE` | SECONDAIRE | 0 | rien à faire |
| `NIVEAU SECONDAIRE` | SECONDAIRE | 0 | même niveau, autre écriture ; un bac saisi ainsi n'est pas signalé |
| `CAP` | SECONDAIRE | 0 | le CAP est un diplôme de niveau secondaire |
| `BAC` | BAC | 1 | rien à faire |
| `NIVEAU BAC` | BAC | 1 | même niveau, autre écriture |
| `NIVEAU UNIVERSITAIRE` | BAC | 1 | imprécis : on sait seulement que la personne a le bac ; pas signalé si le diplôme vaut plus |
| `NIVEAU SUPERIEUR` | BAC | 1 | idem |
| `BAC+1` | BAC+1 | 2 | rien à faire |
| `BAC+2` | BAC+2 | 3 | rien à faire |
| `BAC+2 ET PLUS` | BAC+2 ET PLUS | 3 | gardé tel quel, rang du niveau sûr ; pas signalé si le diplôme vaut BAC+2 ou plus |
| `BAC+3` | BAC+3 | 4 | rien à faire |
| `BAC+3 OU BAC+4` | BAC+3 OU BAC+4 | 4 | gardé tel quel, rang du niveau sûr ; pas signalé si le diplôme vaut BAC+3 ou BAC+4 |
| `BAC+4` | BAC+4 | 5 | rien à faire |
| `BAC+4 ET PLUS` | BAC+4 ET PLUS | 5 | gardé tel quel, rang du niveau sûr ; pas signalé si le diplôme vaut BAC+4 ou plus |
| `BAC+5` | BAC+5 | 6 | rien à faire |
| `BAC+5 ET PLUS` | BAC+5 ET PLUS | 6 | gardé tel quel, rang du niveau sûr ; pas signalé si le diplôme vaut BAC+5 ou plus |
| `>BAC+5` | BAC+5 ET PLUS | 6 | même sens que « BAC+5 et plus » ; saisi surtout sur des masters |
| `BAC+6` | BAC+6 | 7 | gardé tel quel ; pas signalé sur un master (saisi surtout sur des masters) ni sur un doctorat |
| *(vide)* | *(vide)* | — | complété à l'étape 4 à partir du diplôme quand c'est possible |

- Les niveaux imprécis (« et plus », « ou », universitaire, supérieur) sont signalés `NIVEAU_SAISI_IMPRECIS`, à titre d'information.
- Un niveau écrit absent de ce tableau est listé à l'exécution : il faut l'ajouter dans `NIVEAUX_ECRITS`.
- **BAC+7 = doctorat.** Ce niveau n'est pas dans le tableau : il vient du libellé (« Doctorat », étape 3). Les saisies « et plus », « niveau universitaire » ou « Bac + 6 » acceptent jusqu'à BAC+7 : un doctorat saisi « Bac + 5 et plus » n'est donc pas signalé comme une incohérence.
- **EXPERTISE COMPTABLE** : un niveau à part, lui aussi absent du tableau, car il vient du libellé (étape 3). Il a le rang de BAC+5 : une expertise comptable saisie « Bac + 5 » n'est donc pas signalée comme une incohérence, et elle compte comme un BAC+5 quand on cherche le diplôme le plus élevé d'un salarié (étape 8).

In [12]:
# Rang de chaque niveau standard. « Et plus » et « ou » : rang du niveau sûr (BAC+5 ET PLUS = au moins BAC+5).
NIVEAUX = {
    "SECONDAIRE": 0,
    "BAC": 1,
    "BAC+1": 2,
    "BAC+2": 3, "BAC+2 ET PLUS": 3,
    "BAC+3": 4, "BAC+3 OU BAC+4": 4,
    "BAC+4": 5, "BAC+4 ET PLUS": 5,
    "BAC+5": 6, "BAC+5 ET PLUS": 6,
    "EXPERTISE COMPTABLE": 6,            # vient du libellé (étape 3) : un niveau à part, au rang de BAC+5
    "BAC+6": 7, "BAC+6 ET PLUS": 7,
    "BAC+7": 8,                          # doctorat : vient du libellé (étape 3), aucun niveau écrit ne le donne
}
RANG_VERS_NIVEAU = {}                      # rang -> niveau précis (le premier listé pour ce rang)
for niveau, rang in NIVEAUX.items():
    RANG_VERS_NIVEAU.setdefault(rang, niveau)

# Ce qu'on fait de chaque niveau écrit (après l'étape 1), comme dans le tableau ci-dessus :
#   devient  : le niveau standard
#   accepte  : niveaux de diplôme acceptés sans signaler d'incohérence à l'étape 4 (du plus bas au plus haut)
#   imprecis : la saisie ne donne pas un niveau unique (signalée, à titre informatif)
# Pourquoi BAC+7 en haut des saisies « et plus » : un doctorat (BAC+7) saisi « Bac + 5 et plus » reste accepté
#                           devient            accepte de     à                imprecis
NIVEAUX_ECRITS = {
    "SECONDAIRE":           ("SECONDAIRE",     "SECONDAIRE", "SECONDAIRE",    False),
    "NIVEAU SECONDAIRE":    ("SECONDAIRE",     "SECONDAIRE", "BAC",           False),   # via le ficher niveau baccalaureat
    "CAP":                  ("SECONDAIRE",     "SECONDAIRE", "SECONDAIRE",    False),
    "BAC":                  ("BAC",            "BAC",        "BAC",           False),
    "NIVEAU BAC":           ("BAC",            "BAC",        "BAC",           False),
    "NIVEAU UNIVERSITAIRE": ("BAC",            "BAC",        "BAC+7",         True),    # on sait seulement qu'il a le bac
    "NIVEAU SUPERIEUR":     ("BAC",            "BAC",        "BAC+7",         True),
    "BAC+1":                ("BAC+1",          "BAC+1",      "BAC+1",         False),
    "BAC+2":                ("BAC+2",          "BAC+2",      "BAC+2",         False),
    "BAC+2 ET PLUS":        ("BAC+2 ET PLUS",  "BAC+2",      "BAC+7",         True),
    "BAC+3":                ("BAC+3",          "BAC+3",      "BAC+3",         False),
    "BAC+3 OU BAC+4":       ("BAC+3 OU BAC+4", "BAC+3",      "BAC+4",         True),
    "BAC+4":                ("BAC+4",          "BAC+4",      "BAC+4",         False),
    "BAC+4 ET PLUS":        ("BAC+4 ET PLUS",  "BAC+4",      "BAC+7",         True),
    "BAC+5":                ("BAC+5",          "BAC+5",      "BAC+5",         False),
    "BAC+5 ET PLUS":        ("BAC+5 ET PLUS",  "BAC+5",      "BAC+7",         True),
    ">BAC+5":               ("BAC+5 ET PLUS",  "BAC+5",      "BAC+7",         True),    # saisi surtout sur des masters
    "BAC+6":                ("BAC+6",          "BAC+5",      "BAC+7",         False),   # saisi surtout sur des masters ; un doctorat saisi ainsi reste accepté
}
assert all(n in NIVEAUX for ligne in NIVEAUX_ECRITS.values() for n in ligne[:3]), "un niveau de NIVEAUX_ECRITS manque dans NIVEAUX"

ecrit = df["niveau_norm"]
inconnu = ecrit.notna() & ~ecrit.isin(NIVEAUX_ECRITS.keys())
print("Niveaux écrits inconnus (à ajouter dans NIVEAUX_ECRITS) :", sorted(set(ecrit[inconnu])) or "aucun")

ligne = ecrit.map(NIVEAUX_ECRITS)
df["niveau_normalise"] = ligne.str[0]
df["accepte_min"] = ligne.str[1]           # colonnes de vérification, supprimées après l'étape 4
df["accepte_max"] = ligne.str[2]
# Pourquoi eq(True) : un niveau vide compte comme « pas imprécis », comme avant avec fillna(False),
# mais sans l'avertissement que les versions récentes de pandas affichaient sur cette ligne
df["ctrl_niveau_saisi_imprecis"] = ligne.str[3].eq(True) | inconnu

df = df.drop(columns=["niveau_norm"])     # remplacé par niveau_normalise

Niveaux écrits inconnus (à ajouter dans NIVEAUX_ECRITS) : aucun


In [13]:
# Contrôle : chaque écriture source et la valeur retenue
(df.groupby(["niveau", "niveau_normalise"], dropna=False).size()
   .rename("nb_lignes").reset_index()
   .assign(rang=lambda d: d["niveau_normalise"].map(NIVEAUX))
   .sort_values(["rang", "nb_lignes"], ascending=[True, False], na_position="last")
   .drop(columns="rang").reset_index(drop=True))

,niveau,niveau_normalise,nb_lignes
0,NIVEAU SECONDAIRE,SECONDAIRE,2079
1,SECONDAIRE,SECONDAIRE,29
2,CAP,SECONDAIRE,1
3,BAC,BAC,1354
4,Bac,BAC,906
5,NIVEAU UNIVERSITAIRE,BAC,37
6,NIVEAU SUPERIEUR,BAC,24
7,NIVEAU BAC,BAC,8
8,Bac + 1,BAC+1,7
9,Bac +1,BAC+1,6


In [14]:
# Aperçu de la base après l'étape 2
df.head(10)

,matricule,diplome,niveau,specialite,ecole,discipline,ligne_source,diplome_norm,specialite_norm,ecole_norm,discipline_norm,niveau_normalise,accepte_min,accepte_max,ctrl_niveau_saisi_imprecis
0,MATRICULE000001,SECONDAIRE,SECONDAIRE,Autres,LYCEE,NaN,2,SECONDAIRE,None,LYCEE,None,SECONDAIRE,SECONDAIRE,SECONDAIRE,False
1,MATRICULE000002,BREVET TECHNICIEN SPECIALISE,BAC+2,Autres,AUTRES,NaN,3,BREVET TECHNICIEN SPECIALISE,None,None,None,BAC+2,BAC+2,BAC+2,False
2,MATRICULE000002,BACCALAUREAT,BAC,Autres,LYCEE,SCIENCE,4,BACCALAUREAT,None,LYCEE,SCIENCE,BAC,BAC,BAC,False
3,MATRICULE000002,CERTIFICAT D'APTITUDE BANCAIRE,BAC,DOMAINE BANCAIRE,C.F.P.B.,NaN,5,CERTIFICAT D'APTITUDE BANCAIRE,DOMAINE BANCAIRE,C.F.P.B.,None,BAC,BAC,BAC,False
4,MATRICULE000003,DIPLOME DES ETUDES UNIVERSITAIRES GNLE.,BAC+2,DROIT,UNIVMARCDR,DROIT,6,DIPLOME DES ETUDES UNIVERSITAIRES GNLE.,DROIT,UNIVMARCDR,DROIT,BAC+2,BAC+2,BAC+2,False
5,MATRICULE000003,DIPLOME DES ETUDES UNIVERSITAIRES GNLE.,BAC+2,DROIT,UNIVMARCDR,JURIDIQUE,7,DIPLOME DES ETUDES UNIVERSITAIRES GNLE.,DROIT,UNIVMARCDR,JURIDIQUE,BAC+2,BAC+2,BAC+2,False
6,MATRICULE000003,BACCALAUREAT,BAC,LETTRES MODERNES,LYCEE,LETTRES MODERNES,8,BACCALAUREAT,LETTRES MODERNES,LYCEE,LETTRES MODERNES,BAC,BAC,BAC,False
7,MATRICULE000004,SECONDAIRE,SECONDAIRE,Autres,LYCEE,NaN,9,SECONDAIRE,None,LYCEE,None,SECONDAIRE,SECONDAIRE,SECONDAIRE,False
8,MATRICULE000005,INSTITUT DES TECHNIQUES DE BANQUE,BAC+4,DOMAINE BANCAIRE,C.F.P.B.,NaN,10,INSTITUT DES TECHNIQUES DE BANQUE,DOMAINE BANCAIRE,C.F.P.B.,None,BAC+4,BAC+4,BAC+4,False
9,MATRICULE000005,BREVET BANCAIRE,BAC+2,DOMAINE BANCAIRE,C.F.P.B.,NaN,11,BREVET BANCAIRE,DOMAINE BANCAIRE,C.F.P.B.,None,BAC+2,BAC+2,BAC+2,False


---
## Étape 3 : Classification des libellés de diplôme

**Objectif** : pour chaque libellé de diplôme, trouver **ce qu'il est** (`type_diplome` : master, licence, ITB…), le **niveau qu'il vaut** (`niveau_theorique`) et **ce qu'il dit de sa catégorie** (`categorie_libelle`, plus bas).

Le type de diplôme sert seulement à calculer le niveau (étapes 3 et 4) : il n'est pas exporté. La **catégorie** finale (`categorie_diplome` : ingénieur, commerce, université, autres) est faite à l'étape 5, quand on connaît aussi l'école.

**Comment** : des règles par mots-clés, testées dans l'ordre, du plus précis au plus général (« MASTER 1 » avant « MASTER »). Les cas particuliers sont dans `REF_DIPLOMES_MANUEL`, qui passe avant les règles. Chaque niveau a été vérifié contre les niveaux réellement saisis pour ce libellé.

| Niveau | Diplômes |
|---|---|
| BAC+7 | Doctorat |
| EXPERTISE COMPTABLE | Expertise comptable (un niveau à part, au rang de BAC+5) |
| BAC+5 | Master, Master 2, MBA, Executive Master, Ingénieur, Grande école, Classes prépa + grande école, DESS / DEA / DESA, Cycle supérieur |
| BAC+4 | Master 1, Maîtrise, Licence avant 2006, Cycle normal, ITB, Bachelor 4 ans, Ingénieur d'application |
| BAC+3 | Licence, Bachelor 3 ans, Infirmier |
| BAC+2 | DEUG, DUT, BTS, Technicien, Technicien spécialisé, Brevet bancaire |
| BAC | Baccalauréat, Certificat d'aptitude bancaire |
| SECONDAIRE | Secondaire, CAP, formation qualifiante |
| BAC+n | libellé « BAC+n » ou « NIVEAU BAC+n » : le niveau écrit |
| — | Certificat, attestation (pas un diplôme) |

**Libellés à plusieurs niveaux possibles** : le niveau saisi est gardé s'il est compatible avec la fourchette, sinon on prend le plus bas.

| Libellé | Fourchette |
|---|---|
| Technicien spécialisé / Licence pro ; DUT, BTS, licence | BAC+2 à BAC+3 |
| Licence / Master | BAC+3 à BAC+5 |
| Bac+2/3 avec expérience ou Bac+4/5 | BAC+2 à BAC+5 |
| 3ème cycle | BAC+5 à BAC+7 |
| Diplôme des études supérieures (DES) | BAC+4 à BAC+5 (sans saisie : BAC+5) |

**Décisions métier** : Classes prépa / école de commerce = BAC+5 ; ESITH = ingénieur, BAC+5 ; Niveau BAC+2 = BAC+2.

**Sans niveau** (le niveau saisi est gardé) : libellés vagues (`DIPLOME`, `UNIVERSITE`, `Diplôme en…`), domaine seul (`Marketing`…) et diplômes sans équivalence sûre (métiers de banque, degrés comptables, certificat d'études supérieures, CET, brevet professionnel).

**À valider** : Brevet bancaire = BAC+2, ITB = BAC+4, Technicien (non spécialisé) = BAC+2, DES entre BAC+4 et BAC+5.

In [15]:
TYPE_PAR_DEFAUT = "DIPLOME NON IDENTIFIE"

# Décisions libellé par libellé, prioritaires sur les règles (clé = libellé après nettoyage)
# None = aucun niveau théorique justifiable -> on garde le niveau saisi
REF_DIPLOMES_MANUEL = {
    "INFIRMERIE": ("LICENCE", "BAC+3"),                                   # 4 saisies sur 4 BAC+3 ; diplôme d'infirmier = BAC+3
    "INSTITUT SUPERIEUR D'ENTREPRISE": ("LICENCE", "BAC+3"),              # 4 saisies sur 4 BAC+3
    "GRANDES ECOLES DE COMMERCE / MANAGEMENT": ("GRANDE ECOLE (COMMERCE/INGENIEUR)", "BAC+5"),
    "INGENIERIE FISCALE COMPTABLE ET FINANCIERE": ("MASTER", "BAC+5"),    # intitulé de master ; 2 saisies BAC+5
    "ECOLE SUP.DES INDUSTRIES TEXTILE & HABIL": ("INGENIEUR", "BAC+5"),   # ESITH : diplôme d'ingénieur (décision métier)

    # Pas d'équivalence établie ni de saisie probante : on garde le niveau saisi
    "CET SECRETARIAT": ("CERTIFICAT D'ENSEIGNEMENT TECHNIQUE (CET)", None),
    "CERTIFICAT D'ETUDES SUPERIEURES": ("CERTIFICAT D'ETUDES SUPERIEURES", None),   # saisies dispersées de BAC+4 à BAC+6
    "DIPLOME DES ETUDES COMPTABLES SPECIALISE": ("DIPLOME COMPTABLE (DEGRE)", None),

    # Libellés génériques : ils ne disent pas le niveau
    "ACADEMIQUE": ("DIPLOME GENERIQUE (A PRECISER)", None),
    "DIPLOME": ("DIPLOME GENERIQUE (A PRECISER)", None),
    "DIPLOME UNIVERSITAIRE": ("DIPLOME GENERIQUE (A PRECISER)", None),
    "DIPLOME UNIVERSITE": ("DIPLOME GENERIQUE (A PRECISER)", None),
    "UNIVERSITE": ("DIPLOME GENERIQUE (A PRECISER)", None),
}

# (regex, type de diplôme, niveau théorique) : évaluées dans l'ordre sur le texte en MAJUSCULES et SANS ACCENTS,
# la première qui trouve son motif décide. Du plus précis au plus général :
# « MASTER 1 FINANCE » contient « MASTER », la règle Master 1 doit donc passer avant la règle Master.
# Les pourcentages cités sont ceux des niveaux saisis pour le libellé, comptés sans les copies exactes.
REGLES_DIPLOME = [
    # Certificat, attestation : pas un diplôme, donc pas de niveau
    (r"^CERTIFICAT$|^ATTEST|CERTIFICATION|SCRUM MASTER", "CERTIFICATION", None),

    # 1. Libellés qui couvrent plusieurs niveaux -> fourchette (voir FOURCHETTES_NIVEAU)
    (r"TECHNICIEN\s*SPECIALISE\s*/\s*LICENCE|DUT\s*,\s*BTS\s*,\s*LICENCE", "TECHNICIEN SPECIALISE OU LICENCE PRO", None),   # TS = BAC+2, licence pro = BAC+3
    (r"LICENCE\s*/\s*MASTER", "LICENCE OU MASTER", None),
    (r"BAC\s*\+?\s*2\s*/\s*3\s+AVEC\s+.*BAC\s*\+?\s*4\s*/\s*5", "BAC+2 A BAC+5 (DIPLOME NON PRECISE)", None),   # saisies de BAC+3 à BAC+5 : un niveau fixe les écraserait
    (r"(3\s*E(ME)?|TROISIEME) CYCLE", "3EME CYCLE", None),             # DESA / DESS (BAC+5) ou doctorat (BAC+7)

    # 2. Classes prépa puis grande école : BAC+5 (décision métier : la banque ne recrute pas sur une prépa seule)
    (r"CLASSES PREPARATOIRES", "CLASSES PREPA PUIS GRANDE ECOLE", "BAC+5"),

    # 3. Libellé qui commence par BAC+n ou NIVEAU BAC+n : le niveau est écrit, on le prend
    (r"^(NIVEAU )?BAC\s*\+\s*1", "NIVEAU BAC+1 (DIPLOME NON PRECISE)", "BAC+1"),
    (r"^(NIVEAU )?BAC\s*\+\s*2", "NIVEAU BAC+2 (DIPLOME NON PRECISE)", "BAC+2"),
    (r"^(NIVEAU )?BAC\s*\+\s*3", "NIVEAU BAC+3 (DIPLOME NON PRECISE)", "BAC+3"),
    (r"^(NIVEAU )?BAC\s*\+\s*4", "NIVEAU BAC+4 (DIPLOME NON PRECISE)", "BAC+4"),
    (r"^(NIVEAU )?BAC\s*\+\s*5", "NIVEAU BAC+5 (DIPLOME NON PRECISE)", "BAC+5"),
    (r"^(NIVEAU )?BAC\s*\+\s*[6-9]", "NIVEAU BAC+6 ET PLUS (DIPLOME NON PRECISE)", "BAC+6 ET PLUS"),
    (r"^NIVEAU BAC\s*\+", "NIVEAU D'ETUDES (DIPLOME NON PRECISE)", None),     # « NIVEAU BAC+ » sans chiffre lisible

    # 4. Diplômes, par niveau décroissant
    (r"DOCTORAT|^PH\.?\s?D\b|\bPHD\b|DOCTEUR", "DOCTORAT", "BAC+7"),   # doctorat = BAC+7
    (r"\bMASTER\s*-?\s*(1|I)(?![\w.])(?!\s*(ET|/|&|,)\s*(2|II)\b)|\bM1\b|1ERE ANNEE (DE )?MASTER|MASTER\s*\(?\s*1ERE ANNEE", "MASTER 1", "BAC+4"),   # première année de master seule
    (r"\bMASTER\s*-?\s*(2|II)(?![\w.])|\bM2\b|\bMASTER\s*1\s*(ET|/|&|,)\s*2\b", "MASTER", "BAC+5"),   # « MASTER 1 ET 2 » = master complet
    (r"EXECUTIVE MASTER|EXECUTIF MASTER|EXECUTIVE MBA", "MASTER EXECUTIVE", "BAC+5"),
    (r"\bE?MBA\b|M\.B\.A|MASTER OF BUSINESS", "MBA", "BAC+5"),
    (r"\bMASTERE?S?\b|^TYPE MASTER", "MASTER", "BAC+5"),
    # Grande école, sauf si le libellé cite un diplôme plus court (licence, BTS, DUT, prépa, technicien, bachelor)
    (r"^(?!.*\b(LICENCE|BTS|DUT|PREPA|TECHNICIEN|BACH?E?LOR)\b)(ESC\b|.*(GRANDES? ECOLES?|ECOLES? (SUPERIEURE )?DE COMMERCE|ECOLE COMMERCE|ECOLES? D.INGE))", "GRANDE ECOLE (COMMERCE/INGENIEUR)", "BAC+5"),
    (r"INGENIEUR D.APPLICATION", "INGENIEUR D'APPLICATION", "BAC+4"),   # avant « INGENIEUR » : ingénieur d'application = BAC+4
    (r"INGENIEUR|^(DIPLOME D.)?INGENIERIE|^ING\b", "INGENIEUR", "BAC+5"),
    (r"^DIPLOME (DES |D.)ETUDES SUPERIEURE?S\b(?!\s*(SPECIAL|APPLIQU|APPROFONDI))", "DES (DIPLOME DES ETUDES SUPERIEURES)", None),   # deux populations BAC+4 / BAC+5 : fourchette
    (r"EXPERTISE COMPTABLE|EXPERT[ -]COMPTABLE", "EXPERTISE COMPTABLE", "EXPERTISE COMPTABLE"),   # un niveau à part, au rang de BAC+5 (étape 2)
    (r"\bD\.?E\.?S\.?S\.?(?!\w)|\bD\.?E\.?S\.?A\.?(?!\w)|ETUDES SUPERIEURES SPECIAL|ETUDES APPROFONDIES|^DEA\b|ETUDES SUPERIEURES APPLIQUEE|CYCLE SUPERIEUR",
     "DIPLOME 3E CYCLE (DES/DESS/DEA/DESA)", "BAC+5"),
    (r"MAITRISE", "MAITRISE", "BAC+4"),
    (r"LICENCE ANTERIEURE A 2006", "LICENCE (ANCIEN REGIME 4 ANS)", "BAC+4"),   # avant la réforme LMD, la licence durait 4 ans
    (r"CYCLE NORMAL", "DIPLOME CYCLE NORMAL", "BAC+4"),                  # 97,7 % des saisies BAC+4
    (r"INSTITUT (DES )?TECHNIQUES? (DE )?BANQUE|\bITB\b", "ITB", "BAC+4"),   # 99,5 % des saisies BAC+4 (à confirmer CFPB)
    (r"BACH?E?LOR\s*BAC\s*\+\s*4", "BACHELOR 4 ANS", "BAC+4"),
    (r"\d\s*EME DEGRES COMPT", "DIPLOME COMPTABLE (DEGRE)", None),       # pas d'équivalence documentée
    (r"LICENCE|LICENSE|LICENCIE|EQUIVAL\.?LIC|BACH?E?LOR\s*BAC\s*\+\s*3|INFIRM|\bBACH?E?LOR\b", "LICENCE", "BAC+3"),   # BACH?E?LOR couvre aussi la faute « BACHLOR »
    (r"BREVET BANCAIRE", "BREVET BANCAIRE", "BAC+2"),                    # 1 032 saisies BAC+2 sur 1 036 (à confirmer CFPB)
    (r"METIERS DE BANQUE", "DIPLOME DES METIERS DE BANQUE", None),       # aucune saisie sur 25 lignes, pas d'équivalence établie
    # Technicien non spécialisé : BAC+2 selon la convention de la banque (saisies unanimes, 18/18 et 6/6)
    (r"D\.?E\.?U\.?G|ETUDES UNIVERSITAIRES G|\bD\.?U\.?T\b|UNIVERSITAIRE DE TECHNOLOGIE|\bB\.?T\.?S\b|BREVET DE TECHNICIEN SUP|BREVET TECHNICIEN|TECHNICIEN|\bDEUST\b",
     "BAC+2 (DEUG/DUT/BTS/TECHNICIEN)", "BAC+2"),
    (r"APTITUDE BANCAIRE", "CERTIFICAT D'APTITUDE BANCAIRE", "BAC"),     # 94 % des saisies BAC
    (r"BACCALAUREAT|^BAC\b(?!\s*\+)|^NIVEAU BAC$", "BACCALAUREAT", "BAC"),
    (r"BREVET PROFESSIONNEL", "BREVET PROFESSIONNEL", None),             # une seule ligne, rien de solide
    (r"SECONDAIRE|APTITUDE PROFESSIONNELLE|FORMATION PROFESSIONNELLE|FORMATION PRO\b|^CAP\b", "SECONDAIRE / QUALIFIANT", "SECONDAIRE"),

    # 5. Intitulés sans niveau (« Diplôme en… » ou domaine seul) : on garde le niveau saisi
    (r"^DIPLOME EN ", "DIPLOME GENERIQUE (A PRECISER)", None),
    (r"FINANCE|MARKETING|COMMERCE|GESTION|PSYCHOLOGIE|RESSOURCES HUMAINES|COMPTABILITE|AUDIT|BANQUE",
     "DOMAINE SEUL (NIVEAU NON PRECISE)", None),
]

# Types de diplôme à fourchette : (plancher, plafond).
# La saisie est gardée si elle est compatible avec la fourchette (ramenée dedans si besoin), sinon le plancher.
FOURCHETTES_NIVEAU = {
    "TECHNICIEN SPECIALISE OU LICENCE PRO": ("BAC+2", "BAC+3"),
    "LICENCE OU MASTER": ("BAC+3", "BAC+5"),
    "BAC+2 A BAC+5 (DIPLOME NON PRECISE)": ("BAC+2", "BAC+5"),
    "3EME CYCLE": ("BAC+5", "BAC+7"),   # DESS / DEA (BAC+5) jusqu'au doctorat (BAC+7)
    "DES (DIPLOME DES ETUDES SUPERIEURES)": ("BAC+4", "BAC+5"),
}

# Niveau retenu pour un type de diplôme à fourchette quand AUCUN niveau n'est saisi (par défaut : le plancher)
DEFAUT_FOURCHETTE = {
    "DES (DIPLOME DES ETUDES SUPERIEURES)": "BAC+5",   # niveau majoritaire : 78 saisies BAC+5 sur 112 (70 %)
}

In [16]:
def classer_diplome(lib):
    if pd.isna(lib):
        return (None, None, "VIDE")
    if lib in REF_DIPLOMES_MANUEL:             # décision manuelle d'abord
        f, n = REF_DIPLOMES_MANUEL[lib]
        return (f, n, "MANUEL")
    for regex, f, n in REGLES_DIPLOME:         # puis la première règle qui trouve son motif
        if re.search(regex, lib):
            return (f, n, "REGLE")
    return (TYPE_PAR_DEFAUT, None, "NON_CLASSE")

# Classement fait une fois par libellé distinct, puis reporté sur les lignes.
# type_diplome = ce qu'est le diplôme (master, ITB…) ; il sert seulement à calculer le niveau
classement = {lib: classer_diplome(lib) for lib in df["diplome_norm"].dropna().unique()}
df["type_diplome"] = df["diplome_norm"].map(lambda l: classement.get(l, (None,))[0])
df["niveau_theorique"] = df["diplome_norm"].map(lambda l: classement.get(l, (None, None))[1])
df["type_ligne"] = np.where(df["type_diplome"].eq("CERTIFICATION"), "CERTIFICATION", "DIPLOME")   # certification : sans niveau, hors vue salarié

df["ctrl_diplome_manquant"] = df["diplome_norm"].isna()
df["ctrl_libelle_ambigu"] = df["type_diplome"].isin(FOURCHETTES_NIVEAU)
# Libellés qui ne permettent pas d'identifier un diplôme : à relire
df["ctrl_diplome_non_classe"] = df["type_diplome"].isin(
    [TYPE_PAR_DEFAUT, "DIPLOME GENERIQUE (A PRECISER)", "DOMAINE SEUL (NIVEAU NON PRECISE)",
     "NIVEAU D'ETUDES (DIPLOME NON PRECISE)"])

print(f"{len(classement)} libellés distincts classés en {df['type_diplome'].nunique()} types de diplôme")

201 libellés distincts classés en 38 types de diplôme


In [17]:
# Contrôle : les types de diplôme obtenus
(df.groupby(["type_diplome", "niveau_theorique"], dropna=False)
   .agg(libelles_distincts=("diplome_norm", "nunique"), nb_lignes=("ligne_source", "size"))
   .reset_index()
   .assign(rang=lambda d: d["niveau_theorique"].map(NIVEAUX))
   .sort_values(["rang", "nb_lignes"], ascending=[False, False], na_position="last")
   .drop(columns="rang").reset_index(drop=True))

,type_diplome,niveau_theorique,libelles_distincts,nb_lignes
0,DOCTORAT,BAC+7,2,25
1,MASTER,BAC+5,55,4230
2,INGENIEUR,BAC+5,8,743
3,DIPLOME 3E CYCLE (DES/DESS/DEA/DESA),BAC+5,7,350
4,MASTER EXECUTIVE,BAC+5,2,228
5,MBA,BAC+5,4,140
6,GRANDE ECOLE (COMMERCE/INGENIEUR),BAC+5,14,65
7,CLASSES PREPA PUIS GRANDE ECOLE,BAC+5,1,7
8,EXPERTISE COMPTABLE,EXPERTISE COMPTABLE,1,7
9,DIPLOME CYCLE NORMAL,BAC+4,1,692


In [18]:
# Contrôle : libellés que les règles ne savent pas classer (à relire)
(df[df["ctrl_diplome_non_classe"]].groupby(["diplome", "type_diplome"]).size()
   .sort_values(ascending=False).rename("nb_lignes").reset_index())

,diplome,type_diplome,nb_lignes
0,DIPLOME UNIVERSITAIRE,DIPLOME GENERIQUE (A PRECISER),117
1,DIPLOME,DIPLOME GENERIQUE (A PRECISER),94
2,Diplome Université,DIPLOME GENERIQUE (A PRECISER),9
3,Commerce international,DOMAINE SEUL (NIVEAU NON PRECISE),5
4,Diplôme en Audit/Contrôle de Gestion,DIPLOME GENERIQUE (A PRECISER),5
5,Université,DIPLOME GENERIQUE (A PRECISER),5
6,marketing / communication / digital,DOMAINE SEUL (NIVEAU NON PRECISE),5
7,Commerce / finance,DOMAINE SEUL (NIVEAU NON PRECISE),4
8,Finance/Audit/Contrôle et gestion des risques,DOMAINE SEUL (NIVEAU NON PRECISE),4
9,"Finance, Gestion, Management…",DOMAINE SEUL (NIVEAU NON PRECISE),3


**Ce que le libellé dit de la catégorie** (`categorie_libelle`)

La catégorie finale (`categorie_diplome`) est faite à l'étape 5, quand on connaît l'école. Ici, on regarde seulement de quoi parle le libellé :

| Le libellé parle de… | Exemples | Indice |
|---|---|---|
| ingénieur | « Diplôme d'ingénieur », « Ingénieur d'État », « École d'ingénieurs » | DIPLOME D'INGENIEUR |
| commerce | « MBA », « École de commerce », « ESC » | DIPLOME DE COMMERCE |
| université | « Licence », « Master », « Maîtrise », « DEUG », « DUT », « DESS », « Doctorat », « Diplôme universitaire » | DIPLOME UNIVERSITAIRE |
| autre chose | « Baccalauréat », « BTS », « Technicien spécialisé », « ITB », « Brevet bancaire », « Certificat » | AUTRES |

- Le libellé ne parle que d'une de ces choses : c'est son indice.
- Il en cite plusieurs (« Diplôme d'ingénieur / Master école de commerce ») ou aucune (« Diplôme », « Cycle normal ») : pas d'indice, c'est l'école qui décidera.
- Une certification a toujours l'indice AUTRES : ce n'est pas un diplôme.

**Pourquoi « commerce » seul ne suffit pas** : dans « Diplôme en commerce international », c'est le domaine étudié, pas une école de commerce. Il faut « école de commerce », « MBA »…, ou « commerce » cité comme une possibilité parmi d'autres (« Ingénierie / Commerce »).

In [19]:
CAT_ING, CAT_COM, CAT_UNIV, CAT_AUTRES = "DIPLOME D'INGENIEUR", "DIPLOME DE COMMERCE", "DIPLOME UNIVERSITAIRE", "AUTRES"

# De quoi parle le libellé : chaque motif est cherché dans le texte en MAJUSCULES et SANS ACCENTS
INDICES_CATEGORIE = {
    # « Ingénierie fiscale… » ou « … financière » est un master, pas un diplôme d'ingénieur
    CAT_ING: r"INGENIEUR|ECOLES? D.INGE|^ING\b|^(DIPLOME D.)?INGENIERIE\b(?!\s+(FISCALE|FINANCIERE))",
    # « Commerce » seul est un domaine (« commerce international ») : il faut « école de commerce », « MBA »…,
    # ou « commerce » cité comme une possibilité (« ingénierie / commerce », « ingénieur ou de commerce »)
    CAT_COM: r"ECOLES? (SUPERIEURES? )?(DE )?COMMERCE|(/|,|\bOU) (DE )?COMMERC|\bE?MBA\b|M\.B\.A|MASTER OF BUSINESS|BUSINESS SCHOOL|^ESC\b",
    # « Master of business administration » est un MBA, pas un master d'université ; « études supérieurs » (avec la faute) compte aussi
    CAT_UNIV: (r"UNIVERSIT|\bMASTERE?S?\b(?! OF BUSINESS)|\bM[12]\b|LICENCE|LICENSE|LICENCIE|MAITRISE|DOCTORAT|\bPH\.?\s?D\b|DOCTEUR"
               r"|D\.?E\.?U\.?G|\bD\.?U\.?T\b|\bDEUST\b|\bD\.?E\.?S\.?S\b|\bD\.?E\.?A\b|\bD\.?E\.?S\.?A\b|ETUDES SUPERIEURE?S|ETUDES APPROFONDIES"
               r"|(3\s*E(ME)?|TROISIEME) CYCLE|BACH?E?LOR|INFIRM"),
    # Diplômes qui ne sont ni d'ingénieur, ni de commerce, ni d'université
    CAT_AUTRES: (r"BACCALAUREAT|^BAC\b(?!\s*\+)|^NIVEAU BAC$|SECONDAIRE|TECHNICIEN|\bB\.?T\.?S\b|BREVET|APTITUDE|METIERS DE BANQUE"
                 r"|TECHNIQUES? (DE )?BANQUE|\bITB\b|CERTIFICAT|ATTEST|FORMATION PRO|QUALIFIANT|^CAP\b|^CET\b"),
}

def indice_categorie(lib):
    """La catégorie dont parle le libellé, s'il n'en cite qu'une ; sinon None (l'école décidera, étape 5)."""
    if pd.isna(lib):
        return None
    citees = [cat for cat, motif in INDICES_CATEGORIE.items() if re.search(motif, lib)]
    return citees[0] if len(citees) == 1 else None

# Comme pour le type de diplôme : calculé une fois par libellé distinct, puis reporté sur les lignes
indices = {lib: indice_categorie(lib) for lib in df["diplome_norm"].dropna().unique()}
df["categorie_libelle"] = df["diplome_norm"].map(indices)
df.loc[df["type_ligne"].eq("CERTIFICATION"), "categorie_libelle"] = CAT_AUTRES   # une certification n'est pas un diplôme

In [20]:
# Contrôle : indice donné par le libellé (vide = il cite plusieurs catégories ou aucune : l'école décidera à l'étape 5)
display(df["categorie_libelle"].fillna("(aucun indice : l'école décidera)").value_counts().to_frame("nb_lignes"))
# Contrôle : les libellés sans indice les plus fréquents
display(df[df["categorie_libelle"].isna()].groupby("diplome").size()
          .sort_values(ascending=False).head(20).to_frame("nb_lignes"))

,nb_lignes
categorie_libelle,
AUTRES,13224
DIPLOME UNIVERSITAIRE,11396
(aucun indice : l'école décidera),1723
DIPLOME D'INGENIEUR,738
DIPLOME DE COMMERCE,161


,nb_lignes
diplome,
DIPLOME CYCLE NORMAL,692
BAC+3,230
Technicien spécialisé/Licence professionnelle,148
BAC + 2,98
DIPLOME,94
DIPLOME CYCLE SUPERIEUR,92
BAC + 3,65
NIVEAU BAC+2,30
Diplôme Grande Ecole d'Ingénieur/de Commerce,27


In [21]:
# Aperçu de la base après l'étape 3
df.head(10)

,matricule,diplome,niveau,specialite,ecole,discipline,ligne_source,diplome_norm,specialite_norm,ecole_norm,discipline_norm,niveau_normalise,accepte_min,accepte_max,ctrl_niveau_saisi_imprecis,type_diplome,niveau_theorique,type_ligne,ctrl_diplome_manquant,ctrl_libelle_ambigu,ctrl_diplome_non_classe,categorie_libelle
0,MATRICULE000001,SECONDAIRE,SECONDAIRE,Autres,LYCEE,NaN,2,SECONDAIRE,None,LYCEE,None,SECONDAIRE,SECONDAIRE,SECONDAIRE,False,SECONDAIRE / QUALIFIANT,SECONDAIRE,DIPLOME,False,False,False,AUTRES
1,MATRICULE000002,BREVET TECHNICIEN SPECIALISE,BAC+2,Autres,AUTRES,NaN,3,BREVET TECHNICIEN SPECIALISE,None,None,None,BAC+2,BAC+2,BAC+2,False,BAC+2 (DEUG/DUT/BTS/TECHNICIEN),BAC+2,DIPLOME,False,False,False,AUTRES
2,MATRICULE000002,BACCALAUREAT,BAC,Autres,LYCEE,SCIENCE,4,BACCALAUREAT,None,LYCEE,SCIENCE,BAC,BAC,BAC,False,BACCALAUREAT,BAC,DIPLOME,False,False,False,AUTRES
3,MATRICULE000002,CERTIFICAT D'APTITUDE BANCAIRE,BAC,DOMAINE BANCAIRE,C.F.P.B.,NaN,5,CERTIFICAT D'APTITUDE BANCAIRE,DOMAINE BANCAIRE,C.F.P.B.,None,BAC,BAC,BAC,False,CERTIFICAT D'APTITUDE BANCAIRE,BAC,DIPLOME,False,False,False,AUTRES
4,MATRICULE000003,DIPLOME DES ETUDES UNIVERSITAIRES GNLE.,BAC+2,DROIT,UNIVMARCDR,DROIT,6,DIPLOME DES ETUDES UNIVERSITAIRES GNLE.,DROIT,UNIVMARCDR,DROIT,BAC+2,BAC+2,BAC+2,False,BAC+2 (DEUG/DUT/BTS/TECHNICIEN),BAC+2,DIPLOME,False,False,False,DIPLOME UNIVERSITAIRE
5,MATRICULE000003,DIPLOME DES ETUDES UNIVERSITAIRES GNLE.,BAC+2,DROIT,UNIVMARCDR,JURIDIQUE,7,DIPLOME DES ETUDES UNIVERSITAIRES GNLE.,DROIT,UNIVMARCDR,JURIDIQUE,BAC+2,BAC+2,BAC+2,False,BAC+2 (DEUG/DUT/BTS/TECHNICIEN),BAC+2,DIPLOME,False,False,False,DIPLOME UNIVERSITAIRE
6,MATRICULE000003,BACCALAUREAT,BAC,LETTRES MODERNES,LYCEE,LETTRES MODERNES,8,BACCALAUREAT,LETTRES MODERNES,LYCEE,LETTRES MODERNES,BAC,BAC,BAC,False,BACCALAUREAT,BAC,DIPLOME,False,False,False,AUTRES
7,MATRICULE000004,SECONDAIRE,SECONDAIRE,Autres,LYCEE,NaN,9,SECONDAIRE,None,LYCEE,None,SECONDAIRE,SECONDAIRE,SECONDAIRE,False,SECONDAIRE / QUALIFIANT,SECONDAIRE,DIPLOME,False,False,False,AUTRES
8,MATRICULE000005,INSTITUT DES TECHNIQUES DE BANQUE,BAC+4,DOMAINE BANCAIRE,C.F.P.B.,NaN,10,INSTITUT DES TECHNIQUES DE BANQUE,DOMAINE BANCAIRE,C.F.P.B.,None,BAC+4,BAC+4,BAC+4,False,ITB,BAC+4,DIPLOME,False,False,False,AUTRES
9,MATRICULE000005,BREVET BANCAIRE,BAC+2,DOMAINE BANCAIRE,C.F.P.B.,NaN,11,BREVET BANCAIRE,DOMAINE BANCAIRE,C.F.P.B.,None,BAC+2,BAC+2,BAC+2,False,BREVET BANCAIRE,BAC+2,DIPLOME,False,False,False,AUTRES


---
## Étape 4 : Niveau fiabilisé

**Objectif** : produire un niveau unique et fiable par ligne (`niveau_fiabilise`), en croisant le niveau saisi (étape 2) et le niveau théorique (étape 3).

**Règle de cohérence** : la saisie est **compatible** si le niveau du diplôme fait partie des niveaux acceptés pour cette saisie (tableau de l'étape 2). Exemple : un master saisi `BAC+5 ET PLUS` est compatible ; une licence saisie `BAC+5` ne l'est pas.

| Situation | Décision | Contrôle |
|---|---|---|
| Niveau saisi vide, libellé classé | on prend le niveau théorique | `NIVEAU_COMPLETE_PAR_LIBELLE` (informatif) |
| Saisie égale au niveau théorique | rien à faire | — |
| Saisie différente mais compatible | on prend le niveau théorique | `NIVEAU_SAISI_INTERPRETE` (informatif) |
| Saisie incompatible | **le libellé l'emporte** | `INCOHERENCE_NIVEAU` (à vérifier) |
| Libellé à fourchette | saisie conservée telle quelle si elle tient dans la fourchette, ramenée dans la fourchette si elle déborde en partie, sinon plancher ; sans saisie : plancher, ou `DEFAUT_FOURCHETTE` s'il est défini | `LIBELLE_A_FOURCHETTE` ; `INCOHERENCE_NIVEAU` si incompatible |
| Aucun niveau théorique | on garde le niveau saisi (`BAC+6`… ; un « et plus » est arrondi, voir plus bas) | `DIPLOME_NON_CLASSE` si diplôme non identifié |
| Aucun des deux | niveau vide | `NIVEAU_MANQUANT` (à compléter) |
| Certification | pas de niveau | — |


**Choix principal — le libellé prime sur la saisie.** Le libellé vient du diplôme lui-même ; le niveau est une ressaisie manuelle, plus sujette à erreur. Le paramètre `PRIORITE_NIVEAU` permet d'inverser ce choix.

**Arrondi des niveaux « et plus » et « ou »** : le niveau retenu ne garde que ce qui est sûr.

| Niveau avant arrondi | Niveau retenu |
|---|---|
| BAC+2 ET PLUS | BAC+2 |
| BAC+3 OU BAC+4 | BAC+3 |
| BAC+4 ET PLUS | BAC+4 |
| BAC+5 ET PLUS | BAC+5 |
| BAC+6 ET PLUS | BAC+6 |

`BAC+5 ET PLUS` veut dire « au moins BAC+5 » : on retient BAC+5, c'est le seul niveau certain. Ça arrive seulement quand le niveau vient de la saisie (libellé qui ne dit pas le niveau, ou libellé à fourchette) ou d'un libellé « BAC+6 et plus ». La ligne est signalée `NIVEAU_ARRONDI` (informatif) : elle vaut peut-être plus. Ainsi la colonne du niveau n'a que des valeurs simples, prêtes pour les analyses.

**Expertise comptable** : elle garde sa propre valeur de niveau, `EXPERTISE COMPTABLE`, au rang de BAC+5.

**Incohérence fréquente à arbitrer** : des diplômes BAC+2 (surtout Diplôme Technicien Spécialisé, écoles ISTA / OFPPT) sont souvent saisis `Bac + 3`. C'est une question de convention à trancher une fois, plutôt que des erreurs isolées.

In [22]:
# Le libellé prime sur la saisie : il vient du diplôme lui-même, le niveau est ressaisi à la main.
# Mettre "SAISIE" pour inverser ce choix.
PRIORITE_NIVEAU = "DIPLOME"          # "DIPLOME" ou "SAISIE"

saisi, theo = df["niveau_normalise"], df["niveau_theorique"]
est_diplome = df["type_ligne"].eq("DIPLOME")
r_min, r_max = df["accepte_min"].map(NIVEAUX), df["accepte_max"].map(NIVEAUX)
r_theo = theo.map(NIVEAUX)
a_saisie = r_min.notna()

# Saisie compatible = le niveau du diplôme fait partie des niveaux acceptés pour cette saisie (étape 2)
compatible = a_saisie & r_theo.notna() & r_theo.between(r_min, r_max)
df["ctrl_incoherence_niveau"] = a_saisie & r_theo.notna() & ~compatible
df["ctrl_niveau_interprete"] = compatible & saisi.ne(theo)      # ex. master saisi BAC+5 ET PLUS : lu comme BAC+5
df["ctrl_niveau_complete"] = ~a_saisie & r_theo.notna() & est_diplome

if PRIORITE_NIVEAU == "DIPLOME":
    df["niveau_fiabilise"] = theo.where(theo.notna(), saisi)    # sans niveau théorique, la saisie est gardée telle quelle
else:
    # Une saisie imprécise (« NIVEAU UNIVERSITAIRE » = au moins le bac) ne doit pas écraser un libellé plus précis
    precise = saisi.notna() & ~df["ctrl_niveau_saisi_imprecis"]
    df["niveau_fiabilise"] = saisi.where(precise, theo.where(theo.notna(), saisi))
df.loc[~est_diplome, "niveau_fiabilise"] = None                 # certification : pas de niveau

# Libellés à fourchette : la saisie est gardée telle quelle si tous les niveaux qu'elle accepte tiennent dans la fourchette ;
# si elle ne déborde qu'en partie, elle est ramenée dans la fourchette ; incompatible ou absente : le plancher
# (ou le niveau de DEFAUT_FOURCHETTE quand rien n'est saisi)
f_min = df["type_diplome"].map(lambda f: NIVEAUX[FOURCHETTES_NIVEAU[f][0]] if f in FOURCHETTES_NIVEAU else None)
f_max = df["type_diplome"].map(lambda f: NIVEAUX[FOURCHETTES_NIVEAU[f][1]] if f in FOURCHETTES_NIVEAU else None)
a_fourchette = f_min.notna()
bas, haut = np.fmax(r_min, f_min), np.fmin(r_max, f_max)       # partie commune entre la saisie et la fourchette
dans_fourchette = a_fourchette & a_saisie & (bas <= haut)
choix = saisi.map(NIVEAUX).clip(lower=bas, upper=haut)
choix = choix.where(choix.notna(), bas)

df.loc[dans_fourchette, "niveau_fiabilise"] = choix[dans_fourchette].map(RANG_VERS_NIVEAU)
garde_saisie = dans_fourchette & r_min.ge(f_min) & r_max.le(f_max)   # ex. « Bac + 6 » sur un 3e cycle : reste BAC+6
df.loc[garde_saisie, "niveau_fiabilise"] = saisi[garde_saisie]
df.loc[a_fourchette & ~dans_fourchette, "niveau_fiabilise"] = f_min[a_fourchette & ~dans_fourchette].map(RANG_VERS_NIVEAU)
defaut = df["type_diplome"].map(DEFAUT_FOURCHETTE)
df.loc[a_fourchette & ~a_saisie & defaut.notna(), "niveau_fiabilise"] = defaut
df["ctrl_niveau_complete"] |= a_fourchette & ~a_saisie
df["ctrl_niveau_interprete"] |= dans_fourchette & saisi.notna() & df["niveau_fiabilise"].ne(saisi)
df["ctrl_incoherence_niveau"] |= a_fourchette & a_saisie & ~dans_fourchette

# Arrondi au niveau sûr : BAC+5 ET PLUS veut dire « au moins BAC+5 », on retient BAC+5.
# Pourquoi : la colonne du niveau sert aux analyses, elle ne doit avoir que des valeurs simples
# La ligne est signalée NIVEAU_ARRONDI, à titre d'information : elle vaut peut-être plus.
ARRONDI_NIVEAU = {"BAC+2 ET PLUS": "BAC+2", "BAC+3 OU BAC+4": "BAC+3", "BAC+4 ET PLUS": "BAC+4",
                  "BAC+5 ET PLUS": "BAC+5", "BAC+6 ET PLUS": "BAC+6"}
df["ctrl_niveau_arrondi"] = df["niveau_fiabilise"].isin(ARRONDI_NIVEAU)
df["niveau_fiabilise"] = df["niveau_fiabilise"].replace(ARRONDI_NIVEAU)

df["ctrl_niveau_manquant"] = est_diplome & df["niveau_fiabilise"].isna()
df["rang_niveau"] = df["niveau_fiabilise"].map(NIVEAUX)            # sert à trouver le diplôme le plus élevé (étape 8)

df = df.drop(columns=["accepte_min", "accepte_max"])   # cohérence vérifiée, plus utiles

In [23]:
# Contrôle : répartition du niveau avant / après (diplômes uniquement)
ordre = list(NIVEAUX) + ["(vide)"]
avant = df.loc[est_diplome, "niveau_normalise"].fillna("(vide)").value_counts()
theorique = df.loc[est_diplome, "niveau_theorique"].fillna("(vide)").value_counts()
apres = df.loc[est_diplome, "niveau_fiabilise"].fillna("(vide)").value_counts()
pd.DataFrame({"niveau_normalise": avant, "niveau_theorique": theorique, "niveau_fiabilise": apres}).reindex(ordre).fillna(0).astype(int)

,niveau_normalise,niveau_theorique,niveau_fiabilise
SECONDAIRE,2109,30,32
BAC,2325,8320,8327
BAC+1,7,1,1
BAC+2,2995,5629,5837
BAC+2 ET PLUS,2,0,0
BAC+3,2614,4565,4597
BAC+3 OU BAC+4,11,0,0
BAC+4,1222,1923,1979
BAC+4 ET PLUS,12,0,0
BAC+5,3861,5763,6005


In [24]:
# Contrôle : bilan des décisions
pd.Series({
    "Niveaux complétés à partir du libellé": df["ctrl_niveau_complete"].sum(),
    "Saisie différente mais compatible (« et plus », bac saisi « niveau secondaire »…)": df["ctrl_niveau_interprete"].sum(),
    "Incohérences niveau saisi / libellé": df["ctrl_incoherence_niveau"].sum(),
    "Niveaux « et plus » ou « ou » arrondis au niveau sûr": df["ctrl_niveau_arrondi"].sum(),
    "Niveaux encore manquants": df["ctrl_niveau_manquant"].sum(),
}).to_frame("nb_lignes")

,nb_lignes
Niveaux complétés à partir du libellé,11419
"Saisie différente mais compatible (« et plus », bac saisi « niveau secondaire »…)",2364
Incohérences niveau saisi / libellé,254
Niveaux « et plus » ou « ou » arrondis au niveau sûr,13
Niveaux encore manquants,101


In [25]:
# Contrôle : les incohérences les plus fréquentes (le niveau retenu est celui du libellé)
(df[df["ctrl_incoherence_niveau"]]
   .groupby(["diplome", "niveau_normalise", "niveau_theorique"]).size()
   .sort_values(ascending=False).rename("nb_lignes").reset_index().head(25))

,diplome,niveau_normalise,niveau_theorique,nb_lignes
0,DIPLOME TECHNICIEN SPECIALISE,BAC+3,BAC+2,53
1,MAITRISE,BAC+5,BAC+4,11
2,BACCALAUREAT,BAC+2,BAC,8
3,LICENCE ANTERIEURE A 2006,BAC+3,BAC+4,8
4,Master,BAC,BAC+5,8
5,DIPLOME CYCLE SUPERIEUR,BAC+4,BAC+5,7
6,NIVEAU BAC+2,BAC,BAC+2,6
7,DIPLOME CYCLE NORMAL,BAC+3,BAC+4,6
8,Classes Préparatoires / Ecole de Commerce,BAC+3,BAC+5,6
9,TECHNICIEN / OU TECHNICIEN SPECIALISE,BAC+3,BAC+2,6


In [26]:
# Aperçu de la base après l'étape 4
df.head(10)

,matricule,diplome,niveau,specialite,ecole,discipline,ligne_source,diplome_norm,specialite_norm,ecole_norm,discipline_norm,niveau_normalise,ctrl_niveau_saisi_imprecis,type_diplome,niveau_theorique,type_ligne,ctrl_diplome_manquant,ctrl_libelle_ambigu,ctrl_diplome_non_classe,categorie_libelle,ctrl_incoherence_niveau,ctrl_niveau_interprete,ctrl_niveau_complete,niveau_fiabilise,ctrl_niveau_arrondi,ctrl_niveau_manquant,rang_niveau
0,MATRICULE000001,SECONDAIRE,SECONDAIRE,Autres,LYCEE,NaN,2,SECONDAIRE,None,LYCEE,None,SECONDAIRE,False,SECONDAIRE / QUALIFIANT,SECONDAIRE,DIPLOME,False,False,False,AUTRES,False,False,False,SECONDAIRE,False,False,0.0
1,MATRICULE000002,BREVET TECHNICIEN SPECIALISE,BAC+2,Autres,AUTRES,NaN,3,BREVET TECHNICIEN SPECIALISE,None,None,None,BAC+2,False,BAC+2 (DEUG/DUT/BTS/TECHNICIEN),BAC+2,DIPLOME,False,False,False,AUTRES,False,False,False,BAC+2,False,False,3.0
2,MATRICULE000002,BACCALAUREAT,BAC,Autres,LYCEE,SCIENCE,4,BACCALAUREAT,None,LYCEE,SCIENCE,BAC,False,BACCALAUREAT,BAC,DIPLOME,False,False,False,AUTRES,False,False,False,BAC,False,False,1.0
3,MATRICULE000002,CERTIFICAT D'APTITUDE BANCAIRE,BAC,DOMAINE BANCAIRE,C.F.P.B.,NaN,5,CERTIFICAT D'APTITUDE BANCAIRE,DOMAINE BANCAIRE,C.F.P.B.,None,BAC,False,CERTIFICAT D'APTITUDE BANCAIRE,BAC,DIPLOME,False,False,False,AUTRES,False,False,False,BAC,False,False,1.0
4,MATRICULE000003,DIPLOME DES ETUDES UNIVERSITAIRES GNLE.,BAC+2,DROIT,UNIVMARCDR,DROIT,6,DIPLOME DES ETUDES UNIVERSITAIRES GNLE.,DROIT,UNIVMARCDR,DROIT,BAC+2,False,BAC+2 (DEUG/DUT/BTS/TECHNICIEN),BAC+2,DIPLOME,False,False,False,DIPLOME UNIVERSITAIRE,False,False,False,BAC+2,False,False,3.0
5,MATRICULE000003,DIPLOME DES ETUDES UNIVERSITAIRES GNLE.,BAC+2,DROIT,UNIVMARCDR,JURIDIQUE,7,DIPLOME DES ETUDES UNIVERSITAIRES GNLE.,DROIT,UNIVMARCDR,JURIDIQUE,BAC+2,False,BAC+2 (DEUG/DUT/BTS/TECHNICIEN),BAC+2,DIPLOME,False,False,False,DIPLOME UNIVERSITAIRE,False,False,False,BAC+2,False,False,3.0
6,MATRICULE000003,BACCALAUREAT,BAC,LETTRES MODERNES,LYCEE,LETTRES MODERNES,8,BACCALAUREAT,LETTRES MODERNES,LYCEE,LETTRES MODERNES,BAC,False,BACCALAUREAT,BAC,DIPLOME,False,False,False,AUTRES,False,False,False,BAC,False,False,1.0
7,MATRICULE000004,SECONDAIRE,SECONDAIRE,Autres,LYCEE,NaN,9,SECONDAIRE,None,LYCEE,None,SECONDAIRE,False,SECONDAIRE / QUALIFIANT,SECONDAIRE,DIPLOME,False,False,False,AUTRES,False,False,False,SECONDAIRE,False,False,0.0
8,MATRICULE000005,INSTITUT DES TECHNIQUES DE BANQUE,BAC+4,DOMAINE BANCAIRE,C.F.P.B.,NaN,10,INSTITUT DES TECHNIQUES DE BANQUE,DOMAINE BANCAIRE,C.F.P.B.,None,BAC+4,False,ITB,BAC+4,DIPLOME,False,False,False,AUTRES,False,False,False,BAC+4,False,False,5.0
9,MATRICULE000005,BREVET BANCAIRE,BAC+2,DOMAINE BANCAIRE,C.F.P.B.,NaN,11,BREVET BANCAIRE,DOMAINE BANCAIRE,C.F.P.B.,None,BAC+2,False,BREVET BANCAIRE,BAC+2,DIPLOME,False,False,False,AUTRES,False,False,False,BAC+2,False,False,3.0


---
## Étape 5 : École

**Objectif** : remplacer les codes abrégés du SIRH par le **nom réel de l'établissement**, avec son **pays**, sa **nature** (école d'ingénieurs, école de commerce, université…) et un **niveau de confiance**. À la fin de l'étape, la nature de l'école sert à trouver la **catégorie du diplôme**.

**Méthode** : les ~320 écoles distinctes ont été relues une par une et décodées dans un référentiel (`REF_ECOLES`). Quelques exemples :

| Code SIRH | Établissement |
|---|---|
| `UNIVMARCDR` | Université marocaine (non précisée) |
| `IST.SU.C.A.E` | ISCAE |
| `HAUT.ETU.COM.IN` | HECI |
| `EC.NA.SU.IN.A.S` | ENSIAS |
| `EC.MO.ING.RAB` | EMI |
| `CON.NAT.AR.ME` | CNAM |
| `ESC LILLE` | SKEMA Business School |
| `ACESS QUALITY CONSULTING` | Access Quality Consulting (faute corrigée) |

**Confiance**

- `SUR` : décodage certain ;
- `PROBABLE` : décodage très vraisemblable mais pas garanti ;
- `A_CONFIRMER` : code dont l'expansion n'est pas évidente — le nom porte la mention *(A PRECISER)*, à valider par quelqu'un qui connaît l'historique du SIRH.

**Nature**

- `ECOLE_INGENIEURS` : EMI, ENSIAS, EHTP, ENSA, Centrale… ;
- `ECOLE_COMMERCE` : ENCG, ISCAE, HEM, ESSEC… Les écoles qui mêlent informatique et gestion (ISGA, ISGI, ESIG…) y sont rangées : si le libellé dit « ingénieur », c'est lui qui l'emporte ;
- `UNIVERSITE` : universités marocaines et étrangères, publiques ou privées, et ce qui en fait partie (facultés, EST, FST, IUT, CNAM) ;
- `AUTRE` : lycée, OFPPT, CFPB, organismes de formation et de certification ;
- `NON_PRECISE` : on ne sait pas de quel genre d'établissement il s'agit.

**Choix — plus de public / privé.** L'ancien type disait si l'école était publique ou privée, marocaine ou étrangère. Il ne disait pas si elle forme des ingénieurs ou des commerciaux : « grande école publique » mettait ensemble l'ENCG (commerce) et l'EMI (ingénieurs). La nature répond à cette question. Le pays, lui, reste dans sa colonne (`ecole_pays`).

**Choix** : les écoles proches orthographiquement ne sont jamais fusionnées automatiquement (`ENSA` ≠ `ENSAM` ≠ `ENSIAS`). Toute fusion vient du référentiel relu.

Les écoles *génériques* (`LYCEE`, `ENCG` sans ville, `UNIVERSITE MAROCAINE (NON PRECISEE)`) sont conservées mais signalées : l'information existe sans désigner un établissement précis.

In [27]:
# Codes SIRH -> clé du référentiel (étape intermédiaire)
# None = la valeur n'est pas une école (ex. « AUTRES BAC+3 ») -> école vide
ALIAS_CODES_SIRH = {
    "UNIVMARCDR": "UNIVERSITE MAROCAINE (NON PRECISEE)",
    "AUTRE MAIT ETR.": "ETABLISSEMENT ETRANGER (NON PRECISE)",
    "AUTRES BAC+3": None,
    "C.F.P.B.": "CFPB",
    "EC.NAT.COMM.GST": "ENCG (VILLE NON PRECISEE)",
    "ENCG": "ENCG (VILLE NON PRECISEE)",
    "IST.SUP.TEC.AP": "ISTA (OFPPT)",
    "INS.GEST.ADMIN.": "ISGA",
    "EC.SUP.TECH": "EST",
    "HAUT.ETU.MGT": "HEM",
    "SUP CO MARRAK.": "SUP DE CO MARRAKECH",
    "EC.MO.ING.RAB": "EMI",
    "EC.HAS.TRV.PUBL": "EHTP",
}

In [28]:
# Référentiel des écoles : clé -> (nom, pays, nature, confiance), relu école par école
# Confiance : SUR = décodage certain ; PROBABLE = très vraisemblable ; A_CONFIRMER = à valider par quelqu'un qui connaît le SIRH
# Nom None = la valeur saisie n'est pas une école (diplôme ou spécialité saisi au mauvais endroit)
# Nature : le genre d'établissement. C'est elle qui donne la catégorie du diplôme quand le libellé ne suffit pas (fin de l'étape).
# Pourquoi plus de public / privé : on veut savoir si c'est un diplôme d'ingénieur, de commerce ou d'université,
# et l'ancien type ne le disait pas (« grande école publique » mettait ensemble l'ENCG, école de commerce, et l'EMI, école d'ingénieurs).
ING, COM, UNI, AUT, NP = "ECOLE_INGENIEURS", "ECOLE_COMMERCE", "UNIVERSITE", "AUTRE", "NON_PRECISE"
MA, FR, CA, US, GB, BE, CH, ES, DE, JO, AE, SN, TN, DK, AU, HK, INT, NA = (
    "MAROC", "FRANCE", "CANADA", "ETATS-UNIS", "ROYAUME-UNI", "BELGIQUE", "SUISSE", "ESPAGNE",
    "ALLEMAGNE", "JORDANIE", "EMIRATS", "SENEGAL", "TUNISIE", "DANEMARK", "AUSTRALIE", "HONG KONG", "INTERNATIONAL", "")

REF_ECOLES = {
    # ---------------- Écoles d'ingénieurs ----------------
    "EC.NAT.ING": ("ECOLE NATIONALE D'INGENIEURS (NON PRECISEE)", NA, ING, "PROBABLE"),
    "EC.MINES": ("ECOLE DES MINES (NON PRECISEE)", FR, ING, "PROBABLE"),
    "ECOMIN": ("ECOLE DES MINES (NON PRECISEE)", FR, ING, "A_CONFIRMER"),
    "ECOPOLYT.": ("ECOLE POLYTECHNIQUE (NON PRECISEE)", NA, ING, "A_CONFIRMER"),
    "ECOLE SUP ING E": ("ECOLE SUPERIEURE D'INGENIEURS (LIBELLE TRONQUE)", NA, ING, "A_CONFIRMER"),
    "ECOLE SUP ING L": ("ECOLE SUPERIEURE D'INGENIEURS (LIBELLE TRONQUE)", NA, ING, "A_CONFIRMER"),
    "EC SUP ING INF": ("ECOLE SUPERIEURE D'INGENIERIE INFORMATIQUE (NON PRECISEE)", NA, ING, "A_CONFIRMER"),
    "EMI": ("EMI - ECOLE MOHAMMADIA D'INGENIEURS", MA, ING, "SUR"),
    "ENSIAS": ("ENSIAS", MA, ING, "SUR"),
    "EC.NA.SU.IN.A.S": ("ENSIAS", MA, ING, "SUR"),
    "ENSA": ("ENSA (VILLE NON PRECISEE)", MA, ING, "SUR"),
    "ENSA MAROC": ("ENSA (VILLE NON PRECISEE)", MA, ING, "SUR"),
    "ENSEM": ("ENSEM CASABLANCA", MA, ING, "SUR"),
    "ENSAM": ("ENSAM (MEKNES / CASABLANCA)", MA, ING, "PROBABLE"),
    "EHTP": ("EHTP - ECOLE HASSANIA DES TRAVAUX PUBLICS", MA, ING, "SUR"),
    "INSEA / RABAT": ("INSEA RABAT", MA, ING, "SUR"),
    "INPTRAB": ("INPT RABAT", MA, ING, "SUR"),
    "ENIM": ("ENIM - ECOLE NATIONALE DE L'INDUSTRIE MINERALE", MA, ING, "SUR"),
    "IAV": ("IAV HASSAN II", MA, ING, "SUR"),
    "ESI": ("ESI - ECOLE DES SCIENCES DE L'INFORMATION", MA, ING, "PROBABLE"),
    "EC.SCIEN. INFT": ("ESI - ECOLE DES SCIENCES DE L'INFORMATION", MA, ING, "SUR"),
    "ESITH": ("ESITH", MA, ING, "SUR"),
    "ECSUPITH": ("ESITH", MA, ING, "SUR"),
    "AIAC": ("AIAC - ACADEMIE INTERNATIONALE MOHAMMED VI DE L'AVIATION CIVILE", MA, ING, "SUR"),
    "IST.NAT.SCE.AP": ("INSA (FRANCE)", FR, ING, "PROBABLE"),
    "INSA FRANCE": ("INSA (FRANCE)", FR, ING, "SUR"),
    "EMSI": ("EMSI", MA, ING, "SUR"),
    "INS SUP INF APP": ("INSTITUT SUPERIEUR D'INFORMATIQUE APPLIQUEE (A PRECISER)", MA, ING, "A_CONFIRMER"),   # école d'informatique : rangée en ingénieurs
    "IGA": ("IGA - INSTITUT SUPERIEUR DU GENIE APPLIQUE", MA, ING, "SUR"),
    "ESTEM": ("ESTEM CASABLANCA", MA, ING, "SUR"),   # ingénierie et management : rangée en ingénieurs (si le libellé parle de commerce, c'est lui qui l'emporte)
    "HIGH TECH": ("ECOLE HIGH-TECH RABAT", MA, ING, "SUR"),
    "ECP": ("ECOLE CENTRALE PARIS (OU CASABLANCA - A PRECISER)", NA, ING, "A_CONFIRMER"),
    "CENTRALESUPELEC": ("CENTRALESUPELEC", FR, ING, "SUR"),
    "CENTRALE LILLE": ("ECOLE CENTRALE DE LILLE", FR, ING, "SUR"),
    "ECOLE CENTRALE DE LILLE": ("ECOLE CENTRALE DE LILLE", FR, ING, "SUR"),
    "ENPC": ("ECOLE DES PONTS PARISTECH", FR, ING, "SUR"),
    "EC.PONT.CHAUSS": ("ECOLE DES PONTS PARISTECH", FR, ING, "SUR"),
    "ESTP PARIS": ("ESTP PARIS", FR, ING, "SUR"),
    "ENSEEIHT": ("ENSEEIHT TOULOUSE", FR, ING, "SUR"),
    "ENSEEIHT TOULOU": ("ENSEEIHT TOULOUSE", FR, ING, "SUR"),
    "ENSIMAG": ("GRENOBLE INP - ENSIMAG", FR, ING, "SUR"),
    "ENSTA": ("ENSTA PARIS", FR, ING, "SUR"),
    "ENSTAE": ("ENSTA (A PRECISER)", FR, ING, "A_CONFIRMER"),
    "EC.NAT.SUP.TELC": ("TELECOM PARIS (EX ENST)", FR, ING, "PROBABLE"),
    "INT EVRY": ("TELECOM SUDPARIS / IMT-BS (EX INT EVRY)", FR, ING, "SUR"),   # Télécom SudParis (ingénieurs) et IMT-BS (commerce) : rangée en ingénieurs, le libellé tranche s'il parle de commerce
    "ENSAE PARIS": ("ENSAE PARIS", FR, ING, "SUR"),
    "ENSEA": ("ENSEA CERGY", FR, ING, "SUR"),
    "EMN": ("IMT ATLANTIQUE (EX ECOLE DES MINES DE NANTES)", FR, ING, "PROBABLE"),

    # ---------------- Écoles de commerce et de gestion ----------------
    "ECOLE SUP DE MA": ("ECOLE SUPERIEURE DE MANAGEMENT (NON PRECISEE)", NA, COM, "PROBABLE"),
    "EC.SUP.COMME": ("ECOLE SUPERIEURE DE COMMERCE (NON PRECISEE)", NA, COM, "PROBABLE"),
    "EC.SUP.COMMEFR": ("ECOLE SUPERIEURE DE COMMERCE FRANCAISE (NON PRECISEE)", FR, COM, "PROBABLE"),
    "SUP DE CO": ("SUP DE CO (VILLE NON PRECISEE)", NA, COM, "PROBABLE"),
    "INS.SUP.COMMER.": ("INSTITUT SUPERIEUR DE COMMERCE (NON PRECISE)", NA, COM, "PROBABLE"),
    "ECOLE SUPERIEUR INTERNATIONAL DE GEST": ("ECOLE SUPERIEURE INTERNATIONALE DE GESTION (ESIG)", MA, COM, "PROBABLE"),
    "ENCG (VILLE NON PRECISEE)": ("ENCG (VILLE NON PRECISEE)", MA, COM, "SUR"),
    "ENCG SETTAT": ("ENCG SETTAT", MA, COM, "SUR"),
    "ENCG TANGER": ("ENCG TANGER", MA, COM, "SUR"),
    "ENCG CASABLANCA": ("ENCG CASABLANCA", MA, COM, "SUR"),
    "ENCG KENITRA": ("ENCG KENITRA", MA, COM, "SUR"),
    "ENCG AGADIR": ("ENCG AGADIR", MA, COM, "SUR"),
    "ENCG MARRAKECH": ("ENCG MARRAKECH", MA, COM, "SUR"),
    "ENCG OUJDA": ("ENCG OUJDA", MA, COM, "SUR"),
    "ENCG EL JADIDA": ("ENCG EL JADIDA", MA, COM, "SUR"),
    "ENCG FES": ("ENCG FES", MA, COM, "SUR"),
    "ENCG DAKHLA": ("ENCG DAKHLA", MA, COM, "SUR"),
    "LAREFA_ ENCGAGADIR": ("ENCG AGADIR", MA, COM, "PROBABLE"),
    "ISCAE": ("ISCAE", MA, COM, "SUR"),
    "IST.SU.C.A.E": ("ISCAE", MA, COM, "SUR"),
    "INST SUP COM AD": ("ISCAE", MA, COM, "PROBABLE"),
    "INST SUP DE COMM ET ADM DES ESES": ("ISCAE", MA, COM, "SUR"),
    "EC.SU.CO.AD.E": ("ISCAE", MA, COM, "A_CONFIRMER"),
    "ISGA": ("ISGA", MA, COM, "SUR"),   # informatique et gestion : rangée en commerce (si le libellé dit « ingénieur », c'est lui qui l'emporte)
    "HEM": ("HEM", MA, COM, "SUR"),
    "INST.HAU.ETU.MA": ("HEM", MA, COM, "SUR"),
    "IST.MAR.MGT": ("IMM - INSTITUT MAROCAIN DE MANAGEMENT", MA, COM, "PROBABLE"),
    "INSTITUT MAROCAIN DE MANAGEMENT": ("IMM - INSTITUT MAROCAIN DE MANAGEMENT", MA, COM, "SUR"),
    "ESG": ("ESG MAROC", MA, COM, "PROBABLE"),
    "IST.SUP.GST.IN": ("ISGI - INSTITUT SUPERIEUR DE GESTION ET D'INFORMATIQUE", MA, COM, "PROBABLE"),   # gestion et informatique : rangée en commerce, comme ISGA
    "SUP DE CO MARRAKECH": ("SUP DE CO MARRAKECH", MA, COM, "SUR"),
    "HAUT.ETU.COM.IN": ("HECI - HAUTES ETUDES COMMERCIALES ET INFORMATIQUES", MA, COM, "SUR"),   # commerce et informatique : rangée en commerce, comme ISGA
    "HECI": ("HECI - HAUTES ETUDES COMMERCIALES ET INFORMATIQUES", MA, COM, "SUR"),
    "EC.SUP.COM.AFF.": ("ESCA ECOLE DE MANAGEMENT", MA, COM, "SUR"),
    "ESCA": ("ESCA ECOLE DE MANAGEMENT", MA, COM, "SUR"),
    "IST.ADM.ESE": ("INSTITUT SUPERIEUR D'ADMINISTRATION DES ENTREPRISES (A PRECISER)", MA, COM, "A_CONFIRMER"),
    "EC.SUP.IN.GST": ("ESIG - ECOLE SUPERIEURE D'INFORMATIQUE ET DE GESTION", MA, COM, "PROBABLE"),   # informatique et gestion : rangée en commerce, comme ISGA
    "ESIG": ("ESIG - ECOLE SUPERIEURE D'INFORMATIQUE ET DE GESTION", MA, COM, "PROBABLE"),
    "IST.FOR.IN.GST": ("INSTITUT DE FORMATION EN INFORMATIQUE ET GESTION (A PRECISER)", MA, COM, "A_CONFIRMER"),   # informatique et gestion : rangée en commerce, comme ISGA
    "POLYFIN": ("POLYFINANCE", MA, COM, "PROBABLE"),
    "IST.HAUT ETU.EC": ("INSTITUT DES HAUTES ETUDES ECONOMIQUES (A PRECISER)", MA, COM, "A_CONFIRMER"),
    "IHEC": ("IHEC (A PRECISER)", NA, COM, "A_CONFIRMER"),   # IHEC = Institut des hautes études commerciales, quelle que soit la ville
    "INST.SUP.GST": ("ISG - INSTITUT SUPERIEUR DE GESTION (A PRECISER)", MA, COM, "A_CONFIRMER"),
    "SUP MANAGEMENT": ("SUP'MANAGEMENT", MA, COM, "SUR"),
    "HAUT.ETU.COMME": ("HEC MAROC (HAUTES ETUDES COMMERCIALES)", MA, COM, "PROBABLE"),
    "HAUTES ETUDES C": ("HEC MAROC (HAUTES ETUDES COMMERCIALES)", MA, COM, "A_CONFIRMER"),
    "EC.HAUT.ETU.COM": ("HEC MAROC (HAUTES ETUDES COMMERCIALES)", MA, COM, "A_CONFIRMER"),
    "HEC / MAROC": ("HEC MAROC (HAUTES ETUDES COMMERCIALES)", MA, COM, "SUR"),
    "HECB": ("HEC MAROC (HAUTES ETUDES COMMERCIALES)", MA, COM, "A_CONFIRMER"),
    "EC.FR.AFF": ("EFA - ECOLE FRANCAISE DES AFFAIRES", MA, COM, "SUR"),
    "ECOLE FRANCAISE DES AFFAIRE": ("EFA - ECOLE FRANCAISE DES AFFAIRES", MA, COM, "SUR"),
    "EC SUP MAN APP": ("ESMA - ECOLE SUPERIEURE DE MANAGEMENT APPLIQUE", MA, COM, "PROBABLE"),
    "INS SUP ENTREPR": ("INSTITUT SUPERIEUR D'ENTREPRISE (A PRECISER)", MA, COM, "A_CONFIRMER"),
    "INS INF ESES": ("INSTITUT D'INFORMATIQUE DES ENTREPRISES (A PRECISER)", MA, COM, "A_CONFIRMER"),   # informatique de gestion : rangée en commerce, comme ISGA
    "IN.SUP.INFO.COM": ("INSTITUT SUPERIEUR D'INFORMATIQUE ET DE COMMERCE (A PRECISER)", MA, COM, "A_CONFIRMER"),   # informatique et commerce : rangée en commerce, comme ISGA
    "INSTITUT GATE D'INFORMATIQUE ET DE GESTION": ("INSTITUT GATE D'INFORMATIQUE ET DE GESTION", MA, COM, "SUR"),   # informatique et gestion : rangée en commerce, comme ISGA
    "HEC / MONTREAL": ("HEC MONTREAL", CA, COM, "SUR"),
    "HEC / PARIS": ("HEC PARIS", FR, COM, "SUR"),
    "HEC PARIS": ("HEC PARIS", FR, COM, "SUR"),
    "HEC / SUISSE": ("HEC LAUSANNE", CH, COM, "PROBABLE"),   # rattachée à l'université de Lausanne, mais école de commerce par son nom et ses diplômes
    "ESSEC": ("ESSEC BUSINESS SCHOOL", FR, COM, "SUR"),
    "ESSEC BUSINESS SCHOOL": ("ESSEC BUSINESS SCHOOL", FR, COM, "SUR"),
    "ESSEC PARIS": ("ESSEC BUSINESS SCHOOL", FR, COM, "SUR"),
    "ESCP": ("ESCP BUSINESS SCHOOL", FR, COM, "SUR"),
    "ESCP BUSINESS SCHOOL": ("ESCP BUSINESS SCHOOL", FR, COM, "SUR"),
    "EDHEC": ("EDHEC BUSINESS SCHOOL", FR, COM, "SUR"),
    "EM LYON": ("EMLYON BUSINESS SCHOOL", FR, COM, "SUR"),
    "EM GRENOBLE": ("GRENOBLE ECOLE DE MANAGEMENT", FR, COM, "SUR"),
    "AUDENCIA NANTES": ("AUDENCIA NANTES", FR, COM, "SUR"),
    "ESC TOULOUSE": ("TBS EDUCATION (TOULOUSE)", FR, COM, "SUR"),
    "TBS": ("TBS EDUCATION (TOULOUSE)", FR, COM, "SUR"),
    "ESC LILLE": ("SKEMA BUSINESS SCHOOL (EX ESC LILLE)", FR, COM, "SUR"),
    "ESC MONTPELLIER": ("MONTPELLIER BUSINESS SCHOOL", FR, COM, "SUR"),
    "ESC CLE. FERAND": ("ESC CLERMONT BUSINESS SCHOOL", FR, COM, "SUR"),
    "ESC PAU": ("ESC PAU BUSINESS SCHOOL", FR, COM, "SUR"),
    "ESC ROUEN": ("NEOMA BUSINESS SCHOOL (EX ESC ROUEN)", FR, COM, "SUR"),
    "ESC RENNES": ("RENNES SCHOOL OF BUSINESS", FR, COM, "SUR"),
    "ESC AMIENS": ("ESC AMIENS", FR, COM, "SUR"),
    "ESC MARSEILLE": ("KEDGE BUSINESS SCHOOL (EX EUROMED MARSEILLE)", FR, COM, "SUR"),
    "EUROMED MARSEIL": ("KEDGE BUSINESS SCHOOL (EX EUROMED MARSEILLE)", FR, COM, "SUR"),
    "ESCEM POITIERS": ("ESCEM (POITIERS / TOURS)", FR, COM, "SUR"),
    "ESCEM TOURS": ("ESCEM (POITIERS / TOURS)", FR, COM, "SUR"),
    "ICN NANCY": ("ICN BUSINESS SCHOOL (NANCY)", FR, COM, "SUR"),
    "INSEEC PARIS": ("INSEEC (PARIS)", FR, COM, "SUR"),
    "INSEEC BORDEAUX": ("INSEEC (BORDEAUX)", FR, COM, "SUR"),
    "ISGPARIS": ("ISG PARIS", FR, COM, "SUR"),
    "ESGP": ("ESG PARIS", FR, COM, "PROBABLE"),
    "ESGF": ("ESG FINANCE PARIS", FR, COM, "PROBABLE"),
    "INTECPARIS": ("INTEC - CNAM PARIS", FR, COM, "SUR"),   # l'institut de comptabilité et de gestion du CNAM
    "INST CATH. BRUX": ("INSTITUT CATHOLIQUE DES HAUTES ETUDES COMMERCIALES (ICHEC) BRUXELLES", BE, COM, "PROBABLE"),
    "COPENHAGEN BUSINESS SCHOOL": ("COPENHAGEN BUSINESS SCHOOL", DK, COM, "SUR"),   # université publique, mais école de commerce par son nom et ses diplômes

    # ---------------- Universités, et ce qui en fait partie (facultés, EST, FST, IUT, CNAM) ----------------
    "UNIVERSITE MAROCAINE (NON PRECISEE)": ("UNIVERSITE MAROCAINE (NON PRECISEE)", MA, UNI, "SUR"),
    "UNIVERSITE MAROCAINE": ("UNIVERSITE MAROCAINE (NON PRECISEE)", MA, UNI, "SUR"),
    "UNIVERSITE / MAROC": ("UNIVERSITE MAROCAINE (NON PRECISEE)", MA, UNI, "SUR"),
    "UNIV / FRANCE": ("UNIVERSITE FRANCAISE (NON PRECISEE)", FR, UNI, "SUR"),
    "UNIV / CANADA": ("UNIVERSITE CANADIENNE (NON PRECISEE)", CA, UNI, "SUR"),
    "UNIV / USA": ("UNIVERSITE AMERICAINE (NON PRECISEE)", US, UNI, "SUR"),
    "UNIV / SUISSE": ("UNIVERSITE SUISSE (NON PRECISEE)", CH, UNI, "SUR"),
    "UNIGB": ("UNIVERSITE BRITANNIQUE (NON PRECISEE)", GB, UNI, "PROBABLE"),
    "UNIBELGIQUE": ("UNIVERSITE BELGE (NON PRECISEE)", BE, UNI, "PROBABLE"),
    "UNIESPAGNE": ("UNIVERSITE ESPAGNOLE (NON PRECISEE)", ES, UNI, "PROBABLE"),
    "UNIVTU": ("UNIVERSITE TUNISIENNE (NON PRECISEE)", TN, UNI, "A_CONFIRMER"),
    "FACULTE DES SCIENC": ("FACULTE DES SCIENCES (VILLE NON PRECISEE)", MA, UNI, "SUR"),
    "FACULTE DE SCIENCE HUMAINE AIN CHOC": ("FLSH AIN CHOCK - UNIVERSITE HASSAN II CASABLANCA", MA, UNI, "SUR"),
    "ENACTUS FSJES AC-C": ("FSJES AIN CHOCK - UNIVERSITE HASSAN II CASABLANCA (ENACTUS)", MA, UNI, "PROBABLE"),
    "UNIVERSITE HASSAN II DE MOHAMMEDIA": ("UNIVERSITE HASSAN II - MOHAMMEDIA", MA, UNI, "SUR"),
    "EST": ("EST - ECOLE SUPERIEURE DE TECHNOLOGIE (VILLE NON PRECISEE)", MA, UNI, "SUR"),
    "ESTA": ("EST AGADIR", MA, UNI, "PROBABLE"),
    "FST": ("FST - FACULTE DES SCIENCES ET TECHNIQUES (VILLE NON PRECISEE)", MA, UNI, "SUR"),
    "IST.UNIV.TECH": ("IUT - INSTITUT UNIVERSITAIRE DE TECHNOLOGIE", NA, UNI, "PROBABLE"),   # un IUT fait partie d'une université
    "UIR": ("UIR - UNIVERSITE INTERNATIONALE DE RABAT", MA, UNI, "SUR"),   # universités privées : elles ont aussi des écoles d'ingénieurs et de commerce, le libellé tranche alors
    "UNIVERSITE INTERNATIONALE DE RABAT": ("UIR - UNIVERSITE INTERNATIONALE DE RABAT", MA, UNI, "SUR"),
    "UNIVERSIAPOLIS": ("UNIVERSIAPOLIS AGADIR", MA, UNI, "SUR"),
    "AL AKHAWAYN": ("UNIVERSITE AL AKHAWAYN IFRANE", MA, UNI, "SUR"),
    "UIC": ("UIC - UNIVERSITE INTERNATIONALE DE CASABLANCA", MA, UNI, "SUR"),
    "UPF": ("UPF - UNIVERSITE PRIVEE DE FES", MA, UNI, "SUR"),
    "UPM": ("UPM - UNIVERSITE PRIVEE DE MARRAKECH", MA, UNI, "SUR"),
    "UM6P": ("UM6P - UNIVERSITE MOHAMMED VI POLYTECHNIQUE", MA, UNI, "SUR"),
    "ACADEMIE DE SHERBROOK": ("UNIVERSITE DE SHERBROOKE", CA, UNI, "A_CONFIRMER"),
    "DAUPHINE": ("UNIVERSITE PARIS DAUPHINE", FR, UNI, "SUR"),
    "UNIV DAUPHINE": ("UNIVERSITE PARIS DAUPHINE", FR, UNI, "SUR"),
    "UNIV SORBONNE": ("UNIVERSITE SORBONNE (PARIS)", FR, UNI, "SUR"),
    "UNIVERSITE DE CAEN NORMANDIE FRANCE": ("UNIVERSITE DE CAEN NORMANDIE", FR, UNI, "SUR"),
    "UNIVERSITE JEAN MOULIN LYON 3": ("UNIVERSITE JEAN MOULIN LYON 3", FR, UNI, "SUR"),
    "UNIV MONT / CAN": ("UNIVERSITE DE MONTREAL", CA, UNI, "PROBABLE"),
    "CON.NAT.AR.ME": ("CNAM - CONSERVATOIRE NATIONAL DES ARTS ET METIERS", FR, UNI, "SUR"),
    "LONDON SCHOOL OF ECONOMICS AND POLITICAL SCIENCES": ("LONDON SCHOOL OF ECONOMICS (LSE)", GB, UNI, "SUR"),
    "JOHNS HOPKINS UNIVERSITY": ("JOHNS HOPKINS UNIVERSITY", US, UNI, "SUR"),
    "JOHN HOPKINS UNIVERSITY": ("JOHNS HOPKINS UNIVERSITY", US, UNI, "SUR"),
    "UNIVERSITY OF MICHIGAN": ("UNIVERSITY OF MICHIGAN", US, UNI, "SUR"),
    "DEAKIN UNIVERSITY": ("DEAKIN UNIVERSITY", AU, UNI, "SUR"),
    "THE HONG KONG UNIVERSITY OF SCIENCE AND TECHNOLOGY": ("HKUST", HK, UNI, "SUR"),
    "THE OPEN UNIVERSITY ET IAB EUROPE MARKETING": ("THE OPEN UNIVERSITY / IAB EUROPE", GB, UNI, "SUR"),

    # ---------------- Autres : lycée, formation professionnelle ou bancaire, organismes de formation et de certification ----------------
    "LYCEE": ("LYCEE (NON PRECISE)", MA, AUT, "SUR"),
    "E-LEARNING": ("E-LEARNING (PLATEFORME NON PRECISEE)", NA, AUT, "SUR"),
    "ISTA (OFPPT)": ("ISTA - OFPPT", MA, AUT, "SUR"),
    "OFPPT": ("OFPPT", MA, AUT, "SUR"),
    "OFPPTF": ("OFPPT", MA, AUT, "PROBABLE"),
    "L'OFFICE DE LA FORMATION PROFESSIONNELLE ET DE LA PROMOTION DU TRAVAIL": ("OFPPT", MA, AUT, "SUR"),
    "CFPB": ("CFPB - CENTRE DE FORMATION DE LA PROFESSION BANCAIRE", MA, AUT, "SUR"),
    "ATTIJARIWAFA BANK": ("ATTIJARIWAFA BANK (FORMATION INTERNE)", MA, AUT, "SUR"),
    "BMCAF": ("ATTIJARIWAFA BANK (FORMATION INTERNE)", MA, AUT, "A_CONFIRMER"),
    "INJAZ AL MAGHRIB ET FONDATION ATTIJARIWAFA BANK": ("INJAZ AL MAGHRIB / FONDATION ATTIJARIWAFA BANK", MA, AUT, "SUR"),
    "INJAZAL MAGREB": ("INJAZ AL MAGHRIB", MA, AUT, "SUR"),
    "LE CENTRE MAROCAINE DE FINANCE PARTICIPATIVE": ("CENTRE MAROCAIN DE FINANCE PARTICIPATIVE", MA, AUT, "SUR"),
    "AUTORITE MAROCAINE DU MARCHE DES CAPITAUX": ("AMMC - AUTORITE MAROCAINE DU MARCHE DES CAPITAUX", MA, AUT, "SUR"),
    "GENERAL COUNCIL FOR ISLAMIC AND FINANCIAL INSTITUTIONS CIBAFI": ("CIBAFI", INT, AUT, "SUR"),
    "CCI MAROC": ("CHAMBRE DE COMMERCE ET D'INDUSTRIE (MAROC)", MA, AUT, "SUR"),
    "COLLEGE LA SALL": ("COLLEGE LASALLE MAROC", MA, AUT, "PROBABLE"),   # formation technique et professionnelle
    "EC.FR.ENS.TECH.": ("EFET - ECOLE FRANCAISE D'ENSEIGNEMENT TECHNIQUE", MA, AUT, "PROBABLE"),   # formation technique et professionnelle
    "IST.PRO.INFO": ("INSTITUT PROFESSIONNEL D'INFORMATIQUE (A PRECISER)", MA, AUT, "A_CONFIRMER"),   # formation professionnelle
    "ART' COM": ("ART'COM SUP", MA, AUT, "PROBABLE"),   # école de communication et de design : ni ingénieurs, ni commerce, ni université
    "INSTITUT ARABE D'ETUDE JURIDIQUE": ("INSTITUT ARABE D'ETUDES JURIDIQUES", NA, AUT, "SUR"),
    "INSTITUT FRANCAIS DE CASABLANCA IFC": ("INSTITUT FRANCAIS DE CASABLANCA", MA, AUT, "SUR"),
    "SMART GLOBAL UNIVERSITY": ("SMART GLOBAL UNIVERSITY", NA, AUT, "A_CONFIRMER"),
    "ECOLUTION ET PERSPECTIVES": ("ECOLUTION ET PERSPECTIVES (CABINET)", MA, AUT, "A_CONFIRMER"),
    "SUCCESS CLUB DE FORMATION": ("SUCCESS CLUB DE FORMATION", MA, AUT, "SUR"),
    "CENTRE DE COMPETENCE INTERNATIONAL": ("CENTRE DE COMPETENCE INTERNATIONAL", NA, AUT, "A_CONFIRMER"),
    "WATSAN TRAINING DEVELOPEMENT CENTER": ("WATSAN TRAINING DEVELOPMENT CENTER", NA, AUT, "PROBABLE"),
    "T-PROJECT": ("T-PROJECT", NA, AUT, "A_CONFIRMER"),
    "ITAB ACADEMY": ("ITAB ACADEMY", NA, AUT, "A_CONFIRMER"),
    "FORMALIS": ("FORMALIS", NA, AUT, "A_CONFIRMER"),
    "CALLIOPE": ("CALLIOPE", NA, AUT, "A_CONFIRMER"),
    "DELL ET MINISTERE DE LA JEUNESSE ET DES SPORTS": ("DELL / MINISTERE DE LA JEUNESSE ET DES SPORTS", MA, AUT, "SUR"),
    "MINISTERE DE LA JUSTICE ET DES LIBERTES MAROCAIN": ("MINISTERE DE LA JUSTICE (MAROC)", MA, AUT, "SUR"),
    "L'AGENCE NATIONALE DE SECURITE DES SYSTEMES D'INFORMATION": ("ANSSI", FR, AUT, "PROBABLE"),
    "DALE CARNEGIE MAROC": ("DALE CARNEGIE MAROC", MA, AUT, "SUR"),
    "ACCESS QUALITY CONSULTING": ("ACCESS QUALITY CONSULTING", MA, AUT, "SUR"),
    "ACESS QUALITY CONSULTING": ("ACCESS QUALITY CONSULTING", MA, AUT, "SUR"),
    "EFV MANAGEMENT": ("EFV MANAGEMENT", NA, AUT, "PROBABLE"),
    "EFV MANAGMENET": ("EFV MANAGEMENT", NA, AUT, "PROBABLE"),
    "OB2I FINANCE CONSULTING": ("OB2I FINANCE CONSULTING", NA, AUT, "SUR"),
    "LEAD COACH INSTITUT CONVERGENCE": ("LEAD COACH INSTITUT / CONVERGENCE", NA, AUT, "SUR"),
    "LA HAUTE ECOLE DE COACHING FRANCE": ("HAUTE ECOLE DE COACHING (FRANCE)", FR, AUT, "SUR"),
    "ACADEMIE EUROPEENNE DE COACHING": ("ACADEMIE EUROPEENNE DE COACHING", NA, AUT, "SUR"),
    "ACADEMIE CANADIENNE DU SENS": ("ACADEMIE CANADIENNE DU SENS", CA, AUT, "SUR"),
    "INTERNATIONAL NLP": ("INTERNATIONAL NLP", NA, AUT, "PROBABLE"),
    "AMERICAN CANADIAN CENTER FOR HUMAIN DEVELOPMENT": ("AMERICAN CANADIAN CENTER FOR HUMAN DEVELOPMENT", NA, AUT, "SUR"),
    "AMERICAN CANADIAN CENTER FOR HUMAN DEVELOPMENT": ("AMERICAN CANADIAN CENTER FOR HUMAN DEVELOPMENT", NA, AUT, "SUR"),
    "FUNDACION CODESPA": ("FUNDACION CODESPA", ES, AUT, "SUR"),
    "LA FONDATION REINE RANIA JORDANIE": ("FONDATION REINE RANIA (JORDANIE)", JO, AUT, "SUR"),
    "LA FONDATION REINE RANIA - JORDANIE": ("FONDATION REINE RANIA (JORDANIE)", JO, AUT, "SUR"),
    "LA FONDATION REINE RANYA JORDANIE": ("FONDATION REINE RANIA (JORDANIE)", JO, AUT, "SUR"),
    "GOVERNMENT OF DUBAI": ("GOVERNMENT OF DUBAI", AE, AUT, "SUR"),
    "KBS SENEGAL": ("KBS SENEGAL", SN, AUT, "A_CONFIRMER"),
    "BANQUE ALLEMANDE ODDO BHF": ("ODDO BHF", DE, AUT, "SUR"),
    "EUROPEAN BANK FOR RECONSTRUCTION AND DEVELOPMENT": ("BERD", INT, AUT, "SUR"),
    "FMI": ("FMI - FONDS MONETAIRE INTERNATIONAL", INT, AUT, "SUR"),
    "DISASTERREADY.ORG ACCREDITED COURSE BY THE CHARTERED INSTITUTE OF LOGISTICS AND TRANSPORT": ("CILT / DISASTERREADY.ORG", INT, AUT, "SUR"),
    "LONDON SCHOOL OF BUSINESS AND FIANNCE": ("LONDON SCHOOL OF BUSINESS AND FINANCE", GB, AUT, "SUR"),
    "INTERNATIONAL BUSINESS MANAGEMENT INSTITUTE BERLIN.GERMANY": ("IBMI BERLIN", DE, AUT, "SUR"),
    "INTERNATIONAL BUSINESS MANAGEMENT INSTITUTE BERLIN GERMANY": ("IBMI BERLIN", DE, AUT, "SUR"),
    "INTERNATIONAL BUSINESS MANAGEMENT INSTITUTE (IBMI)": ("IBMI BERLIN", DE, AUT, "SUR"),
    "INTERNATIONAL BUSINESS MANAGEMENT INSTITUTE": ("IBMI BERLIN", DE, AUT, "SUR"),
    "HARVARD BUSINESS PUBLISHING": ("HARVARD BUSINESS PUBLISHING", US, AUT, "SUR"),
    "FRANCE EDUCATION INTERNATIONAL": ("FRANCE EDUCATION INTERNATIONAL", FR, AUT, "SUR"),
    "MICROSOFT": ("MICROSOFT", INT, AUT, "SUR"),
    "GOOGLE": ("GOOGLE", INT, AUT, "SUR"),
    "GOOGLE EMEA": ("GOOGLE", INT, AUT, "SUR"),
    "GOOGLE / COURSERA": ("GOOGLE (VIA COURSERA)", INT, AUT, "SUR"),
    "COURSERA": ("COURSERA", INT, AUT, "SUR"),
    "UDEMY": ("UDEMY", INT, AUT, "SUR"),
    "ALISON": ("ALISON", INT, AUT, "SUR"),
    "IBM": ("IBM", INT, AUT, "SUR"),
    "ORACLE": ("ORACLE", INT, AUT, "SUR"),
    "CISCO": ("CISCO", INT, AUT, "SUR"),
    "AWS": ("AMAZON WEB SERVICES", INT, AUT, "SUR"),
    "NOVELL": ("NOVELL", INT, AUT, "SUR"),
    "COMPTIA": ("COMPTIA", INT, AUT, "SUR"),
    "ALTERYX": ("ALTERYX", INT, AUT, "SUR"),
    "SCRUM.ORG": ("SCRUM.ORG", INT, AUT, "SUR"),
    "INTERNATIONAL SCRUM INSTITUTE": ("INTERNATIONAL SCRUM INSTITUTE", INT, AUT, "SUR"),
    "PROJECT MANAGEMENT INSTITUTE": ("PMI - PROJECT MANAGEMENT INSTITUTE", INT, AUT, "SUR"),
    "PECB": ("PECB", INT, AUT, "SUR"),
    "PEOPLECERT": ("PEOPLECERT", INT, AUT, "SUR"),
    "AXELOS": ("AXELOS", INT, AUT, "SUR"),
    "ITIL": ("ITIL (AXELOS / PEOPLECERT)", INT, AUT, "SUR"),
    "EXIN": ("EXIN", INT, AUT, "SUR"),
    "EXINE": ("EXIN", INT, AUT, "PROBABLE"),
    "CERTIPROF": ("CERTIPROF", INT, AUT, "SUR"),
    "ISACA": ("ISACA", INT, AUT, "SUR"),
    "CEH": ("EC-COUNCIL (CEH)", INT, AUT, "SUR"),
    "THE OPEN GROUP": ("THE OPEN GROUP (TOGAF)", INT, AUT, "SUR"),
    "CLOUD NATIVE COMPUTING FOUNDATION (CNCF) - LINUX FOUNDATION": ("CNCF / LINUX FOUNDATION", INT, AUT, "SUR"),
    "INTERNATIONAL DEVOPS CERTIFICATION ACADEMY": ("INTERNATIONAL DEVOPS CERTIFICATION ACADEMY", INT, AUT, "SUR"),
    "CFA INSTITUTE": ("CFA INSTITUTE", INT, AUT, "SUR"),
    "CORPORATE FINANCE INSTITUTE": ("CORPORATE FINANCE INSTITUTE (CFI)", INT, AUT, "SUR"),
    "THE SOCIETY OF TECHNICAL ANALYSTS": ("SOCIETY OF TECHNICAL ANALYSTS", GB, AUT, "SUR"),
    "THE ROYAL INSTITUTION OF CHARTERED SURVEYORS": ("RICS", GB, AUT, "SUR"),
    "VISA BUSINESS SCHOOL": ("VISA BUSINESS SCHOOL", INT, AUT, "SUR"),
    "CAMBRIDGE ENGLISH": ("CAMBRIDGE ENGLISH", GB, AUT, "SUR"),
    "IKI GROUPE CAMBRIGE ENGLISH QUALIFICATION": ("CAMBRIDGE ENGLISH (VIA IKI GROUPE)", GB, AUT, "SUR"),
    "BRITISH COUNCIL": ("BRITISH COUNCIL", GB, AUT, "SUR"),
    "AMIDEAST": ("AMIDEAST", US, AUT, "SUR"),
    "MICROLEARN": ("MICROLEARN", INT, AUT, "SUR"),
    "GREAT LEARNING ACADEMY": ("GREAT LEARNING ACADEMY", INT, AUT, "SUR"),
    "CYBERU": ("CYBERU", INT, AUT, "SUR"),
    "GROVO": ("GROVO", INT, AUT, "SUR"),
    "EJ4": ("EJ4", INT, AUT, "SUR"),
    "CURSA": ("CURSA", INT, AUT, "SUR"),
    "UNOW": ("UNOW", FR, AUT, "SUR"),

    # ---------------- Non précisé : on ne sait pas de quel genre d'établissement il s'agit ----------------
    "ETABLISSEMENT ETRANGER (NON PRECISE)": ("ETABLISSEMENT ETRANGER (NON PRECISE)", NA, NP, "SUR"),
    "AUTRES BAC+2": (None, NA, NP, "SUR"),
    "MASTER BMF": (None, NA, NP, "SUR"),   # c'est un diplôme, pas une école
    "TELECOMMUNICATION": (None, NA, NP, "SUR"),   # c'est une spécialité, pas une école
    "ENA": ("ENA (ADMINISTRATION OU ARCHITECTURE - A PRECISER)", NA, NP, "A_CONFIRMER"),
    "ISIT": ("ISIT (A PRECISER)", MA, NP, "A_CONFIRMER"),
    "ISIM": ("ISIM (A PRECISER)", MA, NP, "A_CONFIRMER"),
    "ESA": ("ESA (A PRECISER)", NA, NP, "A_CONFIRMER"),
    "ISM": ("ISM (A PRECISER)", NA, NP, "A_CONFIRMER"),
    "ISGC": ("ISGC (A PRECISER)", MA, NP, "A_CONFIRMER"),
    "HBF": ("HBF (A PRECISER)", MA, NP, "A_CONFIRMER"),
    "MBCI": ("MBCI (A PRECISER)", MA, NP, "A_CONFIRMER"),
    "EURELEC": ("EURELEC (A PRECISER)", MA, NP, "A_CONFIRMER"),
    "ESNAM": ("ESNAM (A PRECISER)", NA, NP, "A_CONFIRMER"),
    "SUP2TECH": ("SUP2TECH", MA, NP, "PROBABLE"),
    "IMADE": ("IMADE (A PRECISER)", MA, NP, "A_CONFIRMER"),
    "I.E.A.M": ("I.E.A.M (A PRECISER)", NA, NP, "A_CONFIRMER"),
    "IAPM": ("IAPM (A PRECISER)", NA, NP, "A_CONFIRMER"),
    "SIG": ("SIG (A PRECISER)", NA, NP, "A_CONFIRMER"),
    "CMTL": ("CMTL (A PRECISER)", NA, NP, "A_CONFIRMER"),
    "EAGLE": ("EAGLE (A PRECISER)", NA, NP, "A_CONFIRMER"),
    "INPTPAR": ("INPT PARIS (A PRECISER)", FR, NP, "A_CONFIRMER"),
    "ENSIA": ("ENSIA (A PRECISER)", NA, NP, "A_CONFIRMER"),
    "ENAP": ("ENAP (A PRECISER)", NA, NP, "A_CONFIRMER"),
}

In [29]:
def ref_ecole(v):
    if pd.isna(v):
        return (None, "", NP, "SUR")
    cle = ALIAS_CODES_SIRH.get(v, v)
    if cle is None:
        return (None, "", NP, "SUR")
    # École absente du référentiel : gardée telle quelle et à confirmer, jamais rapprochée d'une école voisine (ENSA ≠ ENSAM)
    return REF_ECOLES.get(cle, (cle, "", NP, "A_CONFIRMER"))

res = df["ecole_norm"].map(ref_ecole)
for i, col in enumerate(["ecole_nom", "ecole_pays", "ecole_nature", "ecole_confiance"]):
    df[col] = res.map(lambda t: t[i])

df["ctrl_ecole_manquante"] = df["ecole_nom"].isna()
df["ctrl_ecole_generique"] = df["ecole_nom"].fillna("").str.contains("NON PRECIS")   # l'information existe sans désigner un établissement
df["ctrl_ecole_a_confirmer"] = df["ecole_confiance"].eq("A_CONFIRMER") & df["ecole_nom"].notna()

hors_ref = sorted({ALIAS_CODES_SIRH.get(v, v) for v in df["ecole_norm"].dropna()} - set(REF_ECOLES) - {None})
print("Écoles hors référentiel :", hors_ref or "aucune")

df = df.drop(columns=["ecole_norm"])   # remplacé par ecole_nom

Écoles hors référentiel : aucune


In [30]:
# Contrôle : les décodages les plus fréquents
(df[df["ecole"].notna() & df["ecole_nom"].notna()]
   .groupby(["ecole", "ecole_nom", "ecole_nature", "ecole_confiance"]).size()
   .sort_values(ascending=False).rename("nb_lignes").reset_index().head(30))

,ecole,ecole_nom,ecole_nature,ecole_confiance,nb_lignes
0,LYCEE,LYCEE (NON PRECISE),AUTRE,SUR,7935
1,UNIVMARCDR,UNIVERSITE MAROCAINE (NON PRECISEE),UNIVERSITE,SUR,6014
2,C.F.P.B.,CFPB - CENTRE DE FORMATION DE LA PROFESSION BANCAIRE,AUTRE,SUR,1587
3,IST.SUP.TEC.AP,ISTA - OFPPT,AUTRE,SUR,936
4,OFPPT,OFPPT,AUTRE,SUR,856
5,UNIV/FRANCE,UNIVERSITE FRANCAISE (NON PRECISEE),UNIVERSITE,SUR,642
6,EC.SUP.TECH,EST - ECOLE SUPERIEURE DE TECHNOLOGIE (VILLE NON PRECISEE),UNIVERSITE,SUR,460
7,ENCG SETTAT,ENCG SETTAT,ECOLE_COMMERCE,SUR,332
8,INS.GEST.ADMIN.,ISGA,ECOLE_COMMERCE,SUR,249
9,EC.NAT.COMM.GST,ENCG (VILLE NON PRECISEE),ECOLE_COMMERCE,SUR,237


In [31]:
# Contrôle : répartition par nature d'établissement et par niveau de confiance
display(df["ecole_nature"].value_counts().to_frame("nb_lignes"))
display(df["ecole_confiance"].value_counts().to_frame("nb_lignes"))

,nb_lignes
ecole_nature,
AUTRE,11583
UNIVERSITE,7745
ECOLE_COMMERCE,3919
NON_PRECISE,3117
ECOLE_INGENIEURS,878


,nb_lignes
ecole_confiance,
SUR,25602
PROBABLE,1050
A_CONFIRMER,590


In [32]:
# Contrôle : écoles à confirmer
(df[df["ctrl_ecole_a_confirmer"]].groupby(["ecole", "ecole_nom"]).size()
   .sort_values(ascending=False).rename("nb_lignes").reset_index())

,ecole,ecole_nom,nb_lignes
0,ISGC,ISGC (A PRECISER),100
1,IST.ADM.ESE,INSTITUT SUPERIEUR D'ADMINISTRATION DES ENTREPRISES (A PRECISER),60
2,IST.FOR.IN.GST,INSTITUT DE FORMATION EN INFORMATIQUE ET GESTION (A PRECISER),57
3,HBF,HBF (A PRECISER),47
4,IST.HAUT ETU.EC,INSTITUT DES HAUTES ETUDES ECONOMIQUES (A PRECISER),45
5,INST.SUP.GST,ISG - INSTITUT SUPERIEUR DE GESTION (A PRECISER),40
6,INS SUP INF APP,INSTITUT SUPERIEUR D'INFORMATIQUE APPLIQUEE (A PRECISER),35
7,HAUTES ETUDES C,HEC MAROC (HAUTES ETUDES COMMERCIALES),21
8,EC SUP ING INF,ECOLE SUPERIEURE D'INGENIERIE INFORMATIQUE (NON PRECISEE),21
9,ISIT,ISIT (A PRECISER),18


**Catégorie du diplôme** (`categorie_diplome`)

Chaque ligne reçoit une catégorie : DIPLOME D'INGENIEUR, DIPLOME DE COMMERCE, DIPLOME UNIVERSITAIRE, ou AUTRES quand ce n'est aucun des trois. La première règle qui s'applique décide :

| Ordre | Règle | Exemple |
|---|---|---|
| 1 | L'indice du libellé (étape 3) est ingénieur, commerce ou autres : on le prend | un « Diplôme d'ingénieur » passé dans une université reste un diplôme d'ingénieur ; un « BTS » reste dans AUTRES |
| 2 | Sinon, l'école est une école d'ingénieurs, une école de commerce ou une université : on prend sa nature | un « Master » de l'ISCAE devient un diplôme de commerce |
| 3 | Sinon, l'indice du libellé est université | une « Licence » sans école connue devient un diplôme universitaire |
| 4 | Sinon, l'école est de nature AUTRE (lycée, OFPPT, CFPB, organisme) : AUTRES | un « Technicien spécialisé / licence pro » passé à l'OFPPT |
| 5 | Sinon : AUTRES, et la ligne est signalée `CATEGORIE_NON_TROUVEE` | un « Diplôme » sans école connue |

**Pourquoi l'école passe avant un libellé universitaire** : une licence ou un master se préparent aussi en école de commerce ou d'ingénieurs. L'école dit donc mieux de quel genre de diplôme il s'agit. À l'inverse, « ingénieur » ou « MBA » dans le libellé suffisent : l'école ne les remet pas en cause.

**Pourquoi une ligne peut finir dans AUTRES sans être un bac ou un BTS** : quand ni le libellé ni l'école ne permettent de choisir, par exemple « Diplôme d'ingénieur / Master école de commerce » sans école connue. On ne devine pas : la ligne va dans AUTRES et elle est signalée, pour être complétée à la source si besoin. Ce signalement est informatif : il ne change pas le statut de la ligne.

In [33]:
# Nature de l'école -> catégorie (lycée, formation pro ou bancaire, organismes : aucune des trois)
NATURE_VERS_CATEGORIE = {ING: CAT_ING, COM: CAT_COM, UNI: CAT_UNIV}

indice = df["categorie_libelle"]
par_ecole = df["ecole_nature"].map(NATURE_VERS_CATEGORIE)
regles = [
    indice.isin([CAT_ING, CAT_COM, CAT_AUTRES]),   # 1. le libellé suffit
    par_ecole.notna(),                             # 2. sinon, la nature de l'école
    indice.eq(CAT_UNIV),                           # 3. sinon, le libellé universitaire
    df["ecole_nature"].eq(AUT),                    # 4. sinon, lycée, centre de formation ou organisme
]
df["categorie_diplome"] = np.select(regles, [indice, par_ecole, CAT_UNIV, CAT_AUTRES], default=CAT_AUTRES)
# 5. aucune règle n'a pu choisir : AUTRES par défaut, et la ligne est signalée
df["ctrl_categorie_non_trouvee"] = ~np.logical_or.reduce(regles)

df = df.drop(columns=["type_diplome", "categorie_libelle"])   # niveau (étape 4) et catégorie faits : plus utiles

In [34]:
# Contrôle : catégorie obtenue pour chaque niveau, du plus haut au plus bas
(pd.crosstab(df["niveau_fiabilise"].fillna("(vide)"), df["categorie_diplome"], margins=True, margins_name="Total")
   .reindex(list(NIVEAUX)[::-1] + ["(vide)", "Total"]).dropna(how="all").astype(int))

categorie_diplome,AUTRES,DIPLOME D'INGENIEUR,DIPLOME DE COMMERCE,DIPLOME UNIVERSITAIRE,Total
niveau_fiabilise,,,,,
BAC+7,0,2,3,20,25
BAC+6,0,0,3,4,7
EXPERTISE COMPTABLE,2,0,4,1,7
BAC+5,25,896,2602,2482,6005
BAC+4,395,45,750,789,1979
BAC+3,98,44,284,4171,4597
BAC+2,4513,4,18,1302,5837
BAC+1,0,0,0,1,1
BAC,8325,0,0,2,8327


In [35]:
# Contrôle : lignes dont la catégorie n'a pas été trouvée (rangées dans AUTRES), les plus fréquentes
(df[df["ctrl_categorie_non_trouvee"]].groupby(["diplome", "ecole_nom"], dropna=False).size()
   .sort_values(ascending=False).rename("nb_lignes").reset_index().head(20))

,diplome,ecole_nom,nb_lignes
0,DIPLOME CYCLE NORMAL,NaN,74
1,DIPLOME CYCLE NORMAL,ISGC (A PRECISER),53
2,BAC+3,NaN,27
3,BAC + 3,NaN,20
4,DIPLOME,NaN,19
5,BAC+3,HBF (A PRECISER),18
6,BAC + 2,NaN,17
7,Technicien spécialisé/Licence professionnelle,NaN,16
8,NIVEAU BAC+2,NaN,9
9,DIPLOME CYCLE NORMAL,ISIT (A PRECISER),7


In [36]:
# Aperçu de la base après l'étape 5
df.head(10)

,matricule,diplome,niveau,specialite,ecole,discipline,ligne_source,diplome_norm,specialite_norm,discipline_norm,niveau_normalise,ctrl_niveau_saisi_imprecis,niveau_theorique,type_ligne,ctrl_diplome_manquant,ctrl_libelle_ambigu,ctrl_diplome_non_classe,ctrl_incoherence_niveau,ctrl_niveau_interprete,ctrl_niveau_complete,niveau_fiabilise,ctrl_niveau_arrondi,ctrl_niveau_manquant,rang_niveau,ecole_nom,ecole_pays,ecole_nature,ecole_confiance,ctrl_ecole_manquante,ctrl_ecole_generique,ctrl_ecole_a_confirmer,categorie_diplome,ctrl_categorie_non_trouvee
0,MATRICULE000001,SECONDAIRE,SECONDAIRE,Autres,LYCEE,NaN,2,SECONDAIRE,None,None,SECONDAIRE,False,SECONDAIRE,DIPLOME,False,False,False,False,False,False,SECONDAIRE,False,False,0.0,LYCEE (NON PRECISE),MAROC,AUTRE,SUR,False,True,False,AUTRES,False
1,MATRICULE000002,BREVET TECHNICIEN SPECIALISE,BAC+2,Autres,AUTRES,NaN,3,BREVET TECHNICIEN SPECIALISE,None,None,BAC+2,False,BAC+2,DIPLOME,False,False,False,False,False,False,BAC+2,False,False,3.0,None,,NON_PRECISE,SUR,True,False,False,AUTRES,False
2,MATRICULE000002,BACCALAUREAT,BAC,Autres,LYCEE,SCIENCE,4,BACCALAUREAT,None,SCIENCE,BAC,False,BAC,DIPLOME,False,False,False,False,False,False,BAC,False,False,1.0,LYCEE (NON PRECISE),MAROC,AUTRE,SUR,False,True,False,AUTRES,False
3,MATRICULE000002,CERTIFICAT D'APTITUDE BANCAIRE,BAC,DOMAINE BANCAIRE,C.F.P.B.,NaN,5,CERTIFICAT D'APTITUDE BANCAIRE,DOMAINE BANCAIRE,None,BAC,False,BAC,DIPLOME,False,False,False,False,False,False,BAC,False,False,1.0,CFPB - CENTRE DE FORMATION DE LA PROFESSION BANCAIRE,MAROC,AUTRE,SUR,False,False,False,AUTRES,False
4,MATRICULE000003,DIPLOME DES ETUDES UNIVERSITAIRES GNLE.,BAC+2,DROIT,UNIVMARCDR,DROIT,6,DIPLOME DES ETUDES UNIVERSITAIRES GNLE.,DROIT,DROIT,BAC+2,False,BAC+2,DIPLOME,False,False,False,False,False,False,BAC+2,False,False,3.0,UNIVERSITE MAROCAINE (NON PRECISEE),MAROC,UNIVERSITE,SUR,False,True,False,DIPLOME UNIVERSITAIRE,False
5,MATRICULE000003,DIPLOME DES ETUDES UNIVERSITAIRES GNLE.,BAC+2,DROIT,UNIVMARCDR,JURIDIQUE,7,DIPLOME DES ETUDES UNIVERSITAIRES GNLE.,DROIT,JURIDIQUE,BAC+2,False,BAC+2,DIPLOME,False,False,False,False,False,False,BAC+2,False,False,3.0,UNIVERSITE MAROCAINE (NON PRECISEE),MAROC,UNIVERSITE,SUR,False,True,False,DIPLOME UNIVERSITAIRE,False
6,MATRICULE000003,BACCALAUREAT,BAC,LETTRES MODERNES,LYCEE,LETTRES MODERNES,8,BACCALAUREAT,LETTRES MODERNES,LETTRES MODERNES,BAC,False,BAC,DIPLOME,False,False,False,False,False,False,BAC,False,False,1.0,LYCEE (NON PRECISE),MAROC,AUTRE,SUR,False,True,False,AUTRES,False
7,MATRICULE000004,SECONDAIRE,SECONDAIRE,Autres,LYCEE,NaN,9,SECONDAIRE,None,None,SECONDAIRE,False,SECONDAIRE,DIPLOME,False,False,False,False,False,False,SECONDAIRE,False,False,0.0,LYCEE (NON PRECISE),MAROC,AUTRE,SUR,False,True,False,AUTRES,False
8,MATRICULE000005,INSTITUT DES TECHNIQUES DE BANQUE,BAC+4,DOMAINE BANCAIRE,C.F.P.B.,NaN,10,INSTITUT DES TECHNIQUES DE BANQUE,DOMAINE BANCAIRE,None,BAC+4,False,BAC+4,DIPLOME,False,False,False,False,False,False,BAC+4,False,False,5.0,CFPB - CENTRE DE FORMATION DE LA PROFESSION BANCAIRE,MAROC,AUTRE,SUR,False,False,False,AUTRES,False
9,MATRICULE000005,BREVET BANCAIRE,BAC+2,DOMAINE BANCAIRE,C.F.P.B.,NaN,11,BREVET BANCAIRE,DOMAINE BANCAIRE,None,BAC+2,False,BAC+2,DIPLOME,False,False,False,False,False,False,BAC+2,False,False,3.0,CFPB - CENTRE DE FORMATION DE LA PROFESSION BANCAIRE,MAROC,AUTRE,SUR,False,False,False,AUTRES,False


---
## Étape 6 : Dédoublonnage

**Objectif** : supprimer les lignes répétées par l'extraction, sans perdre d'information.

**Choix — dédoublonner après la normalisation, pas avant.** Avant normalisation, `Bac + 5` et `BAC+5`, ou `IST.SU.C.A.E` et `ISCAE`, empêchent de reconnaître deux lignes identiques. La clé de dédoublonnage porte donc sur les valeurs nettoyées ou fiabilisées : matricule, libellé de diplôme, niveau fiabilisé, école, et la spécialité telle que saisie (ou la discipline quand la spécialité est vide). La spécialité reste dans la clé pour ne pas supprimer deux diplômes qui ne diffèrent que par elle (75 lignes sur cette extraction, par exemple deux masters de spécialités différentes dans la même école).

Les lignes supprimées ne disparaissent pas : elles sont exportées dans l'onglet `Doublons_supprimes` avec leur numéro de ligne d'origine.

**Quasi-doublons** : même salarié, même diplôme, même niveau, mais spécialité ou école différente. On ne peut pas savoir laquelle est la bonne sans le dossier : les deux lignes sont **conservées** et signalées `QUASI_DOUBLON_A_ARBITRER`.

In [37]:
# Spécialité telle que saisie (nettoyée) ; la discipline la remplace seulement quand elle est vide.
# Elle reste dans la clé pour ne pas supprimer deux diplômes qui ne diffèrent que par la spécialité
# (ex. deux masters de spécialités différentes dans la même école : 75 lignes sur cette extraction).
specialite_cle = df["specialite_norm"].fillna(df["discipline_norm"])

# Clé sur les valeurs nettoyées : « Bac + 5 » et « BAC+5 » ne doivent pas empêcher de reconnaître un doublon
CLES_DOUBLON = ["matricule", "diplome_norm", "niveau_fiabilise", "specialite_cle", "ecole_nom"]

nb_avant = len(df)
doublons_bruts = df.duplicated(subset=COLS).sum()
dup = df.assign(specialite_cle=specialite_cle).duplicated(subset=CLES_DOUBLON, keep="first")
doublons = df[dup].copy()              # gardés pour l'onglet Doublons_supprimes : rien ne disparaît
df = df[~dup].copy()

# Même salarié, même diplôme, même niveau, mais spécialité ou école différente : sans le dossier on ne sait pas
# laquelle est la bonne, les deux lignes sont gardées et signalées
df["ctrl_quasi_doublon"] = (
    df.duplicated(subset=["matricule", "diplome_norm", "niveau_fiabilise"], keep=False)
    & df["type_ligne"].eq("DIPLOME")
)

pd.Series({
    "Lignes avant": nb_avant,
    "Doublons détectés sur les valeurs brutes": doublons_bruts,
    "Doublons détectés après normalisation (supprimés)": len(doublons),
    "Lignes après": len(df),
    "Lignes en quasi-doublon (conservées, à arbitrer)": df["ctrl_quasi_doublon"].sum(),
}).to_frame("valeur")

,valeur
Lignes avant,27242
Doublons détectés sur les valeurs brutes,6389
Doublons détectés après normalisation (supprimés),6408
Lignes après,20834
"Lignes en quasi-doublon (conservées, à arbitrer)",923


In [38]:
# Contrôle : exemple de quasi-doublon
m = df.loc[df["ctrl_quasi_doublon"], "matricule"].iloc[0]
df.loc[df["matricule"].eq(m), ["matricule", "diplome", "niveau_fiabilise", "specialite", "ecole_nom"]]

,matricule,diplome,niveau_fiabilise,specialite,ecole_nom
276,MATRICULE000157,DIPLOME CYCLE NORMAL,BAC+4,MANAGEMENT ENTREPRISES,None
277,MATRICULE000157,BREVET BANCAIRE,BAC+2,DOMAINE BANCAIRE,CFPB - CENTRE DE FORMATION DE LA PROFESSION BANCAIRE
278,MATRICULE000157,DIPLOME CYCLE NORMAL,BAC+4,FINANCE-MANAGEMENT,None
279,MATRICULE000157,DIPLOME UNIVERSITAIRE DE TECHNOLOGIE,BAC+2,Autres,None


In [39]:
# Aperçu de la base après l'étape 6
df.head(10)

,matricule,diplome,niveau,specialite,ecole,discipline,ligne_source,diplome_norm,specialite_norm,discipline_norm,niveau_normalise,ctrl_niveau_saisi_imprecis,niveau_theorique,type_ligne,ctrl_diplome_manquant,ctrl_libelle_ambigu,ctrl_diplome_non_classe,ctrl_incoherence_niveau,ctrl_niveau_interprete,ctrl_niveau_complete,niveau_fiabilise,ctrl_niveau_arrondi,ctrl_niveau_manquant,rang_niveau,ecole_nom,ecole_pays,ecole_nature,ecole_confiance,ctrl_ecole_manquante,ctrl_ecole_generique,ctrl_ecole_a_confirmer,categorie_diplome,ctrl_categorie_non_trouvee,ctrl_quasi_doublon
0,MATRICULE000001,SECONDAIRE,SECONDAIRE,Autres,LYCEE,NaN,2,SECONDAIRE,None,None,SECONDAIRE,False,SECONDAIRE,DIPLOME,False,False,False,False,False,False,SECONDAIRE,False,False,0.0,LYCEE (NON PRECISE),MAROC,AUTRE,SUR,False,True,False,AUTRES,False,False
1,MATRICULE000002,BREVET TECHNICIEN SPECIALISE,BAC+2,Autres,AUTRES,NaN,3,BREVET TECHNICIEN SPECIALISE,None,None,BAC+2,False,BAC+2,DIPLOME,False,False,False,False,False,False,BAC+2,False,False,3.0,None,,NON_PRECISE,SUR,True,False,False,AUTRES,False,False
2,MATRICULE000002,BACCALAUREAT,BAC,Autres,LYCEE,SCIENCE,4,BACCALAUREAT,None,SCIENCE,BAC,False,BAC,DIPLOME,False,False,False,False,False,False,BAC,False,False,1.0,LYCEE (NON PRECISE),MAROC,AUTRE,SUR,False,True,False,AUTRES,False,False
3,MATRICULE000002,CERTIFICAT D'APTITUDE BANCAIRE,BAC,DOMAINE BANCAIRE,C.F.P.B.,NaN,5,CERTIFICAT D'APTITUDE BANCAIRE,DOMAINE BANCAIRE,None,BAC,False,BAC,DIPLOME,False,False,False,False,False,False,BAC,False,False,1.0,CFPB - CENTRE DE FORMATION DE LA PROFESSION BANCAIRE,MAROC,AUTRE,SUR,False,False,False,AUTRES,False,False
4,MATRICULE000003,DIPLOME DES ETUDES UNIVERSITAIRES GNLE.,BAC+2,DROIT,UNIVMARCDR,DROIT,6,DIPLOME DES ETUDES UNIVERSITAIRES GNLE.,DROIT,DROIT,BAC+2,False,BAC+2,DIPLOME,False,False,False,False,False,False,BAC+2,False,False,3.0,UNIVERSITE MAROCAINE (NON PRECISEE),MAROC,UNIVERSITE,SUR,False,True,False,DIPLOME UNIVERSITAIRE,False,False
6,MATRICULE000003,BACCALAUREAT,BAC,LETTRES MODERNES,LYCEE,LETTRES MODERNES,8,BACCALAUREAT,LETTRES MODERNES,LETTRES MODERNES,BAC,False,BAC,DIPLOME,False,False,False,False,False,False,BAC,False,False,1.0,LYCEE (NON PRECISE),MAROC,AUTRE,SUR,False,True,False,AUTRES,False,False
7,MATRICULE000004,SECONDAIRE,SECONDAIRE,Autres,LYCEE,NaN,9,SECONDAIRE,None,None,SECONDAIRE,False,SECONDAIRE,DIPLOME,False,False,False,False,False,False,SECONDAIRE,False,False,0.0,LYCEE (NON PRECISE),MAROC,AUTRE,SUR,False,True,False,AUTRES,False,False
8,MATRICULE000005,INSTITUT DES TECHNIQUES DE BANQUE,BAC+4,DOMAINE BANCAIRE,C.F.P.B.,NaN,10,INSTITUT DES TECHNIQUES DE BANQUE,DOMAINE BANCAIRE,None,BAC+4,False,BAC+4,DIPLOME,False,False,False,False,False,False,BAC+4,False,False,5.0,CFPB - CENTRE DE FORMATION DE LA PROFESSION BANCAIRE,MAROC,AUTRE,SUR,False,False,False,AUTRES,False,False
9,MATRICULE000005,BREVET BANCAIRE,BAC+2,DOMAINE BANCAIRE,C.F.P.B.,NaN,11,BREVET BANCAIRE,DOMAINE BANCAIRE,None,BAC+2,False,BAC+2,DIPLOME,False,False,False,False,False,False,BAC+2,False,False,3.0,CFPB - CENTRE DE FORMATION DE LA PROFESSION BANCAIRE,MAROC,AUTRE,SUR,False,False,False,AUTRES,False,False
10,MATRICULE000005,DIPLOME DES ETUDES UNIVERSITAIRES GNLE.,BAC+2,GESTION DES ENTREPRISES,UNIVMARCDR,GESTION,12,DIPLOME DES ETUDES UNIVERSITAIRES GNLE.,GESTION DES ENTREPRISES,GESTION,BAC+2,False,BAC+2,DIPLOME,False,False,False,False,False,False,BAC+2,False,False,3.0,UNIVERSITE MAROCAINE (NON PRECISEE),MAROC,UNIVERSITE,SUR,False,True,False,DIPLOME UNIVERSITAIRE,False,False


---
## Étape 7 : Contrôles et statut par ligne

**Objectif** : que chaque ligne dise clairement si elle est exploitable, et sinon ce qu'il faut faire.

Les contrôles calculés aux étapes précédentes (`ctrl_*`) sont de deux natures :

- **informatifs** : une transformation a été faite, la ligne reste exploitable (niveau complété, saisie interprétée ou imprécise, école générique, catégorie non trouvée) ;
- **bloquants** : une décision humaine est nécessaire.

Le **statut** retient le motif bloquant le plus important, dans cet ordre de priorité :

1. `A COMPLETER - diplôme`
2. `A COMPLETER - niveau`
3. `A VERIFIER - niveau saisi ≠ libellé`
4. `A ARBITRER - quasi-doublon`
5. `A CONFIRMER - école`
6. sinon `OK`

In [40]:
# Bloquant : une décision humaine est nécessaire. Informatif : une transformation a été faite, la ligne reste exploitable.
CONTROLES = {
    "ctrl_diplome_manquant":          ("DIPLOME_MANQUANT", "bloquant"),
    "ctrl_niveau_manquant":           ("NIVEAU_MANQUANT", "bloquant"),
    "ctrl_incoherence_niveau":        ("INCOHERENCE_NIVEAU", "bloquant"),
    "ctrl_quasi_doublon":             ("QUASI_DOUBLON_A_ARBITRER", "bloquant"),
    "ctrl_ecole_a_confirmer":         ("ECOLE_A_CONFIRMER", "bloquant"),
    "ctrl_diplome_non_classe":        ("DIPLOME_NON_CLASSE", "informatif"),
    "ctrl_libelle_ambigu":            ("LIBELLE_A_FOURCHETTE", "informatif"),
    "ctrl_niveau_saisi_imprecis":     ("NIVEAU_SAISI_IMPRECIS", "informatif"),
    "ctrl_niveau_complete":           ("NIVEAU_COMPLETE_PAR_LIBELLE", "informatif"),
    "ctrl_niveau_interprete":         ("NIVEAU_SAISI_INTERPRETE", "informatif"),
    "ctrl_niveau_arrondi":            ("NIVEAU_ARRONDI", "informatif"),
    "ctrl_ecole_manquante":           ("ECOLE_MANQUANTE", "informatif"),
    "ctrl_ecole_generique":           ("ECOLE_GENERIQUE", "informatif"),
    "ctrl_categorie_non_trouvee":     ("CATEGORIE_NON_TROUVEE", "informatif"),
}
for c in CONTROLES:
    df[c] = df[c].fillna(False).astype(bool)

df["flags"] = df[list(CONTROLES)].apply(
    lambda r: " | ".join(CONTROLES[c][0] for c in CONTROLES if r[c]), axis=1)

# Statut = le motif bloquant le plus important, dans l'ordre de priorité ci-dessous.
# Pourquoi le diplôme manquant en premier : DIPLOME_MANQUANT est bloquant, mais il n'avait pas de statut,
# donc une ligne sans libellé de diplôme sortait « OK ». Sans diplôme, il n'y a rien d'autre à vérifier avant.
df["statut"] = np.select(
    [df["ctrl_diplome_manquant"], df["ctrl_niveau_manquant"], df["ctrl_incoherence_niveau"],
     df["ctrl_quasi_doublon"], df["ctrl_ecole_a_confirmer"]],
    ["A COMPLETER - diplôme", "A COMPLETER - niveau", "A VERIFIER - niveau saisi ≠ libellé",
     "A ARBITRER - quasi-doublon", "A CONFIRMER - école"],
    default="OK",
)

In [41]:
# Contrôle : nombre de lignes par contrôle
tab_controles = pd.DataFrame(
    [(code_ctrl, nature, int(df[c].sum())) for c, (code_ctrl, nature) in CONTROLES.items()],
    columns=["controle", "nature", "nb_lignes"],
).sort_values(["nature", "nb_lignes"], ascending=[True, False]).reset_index(drop=True)
tab_controles

,controle,nature,nb_lignes
0,QUASI_DOUBLON_A_ARBITRER,bloquant,923
1,ECOLE_A_CONFIRMER,bloquant,459
2,INCOHERENCE_NIVEAU,bloquant,191
3,NIVEAU_MANQUANT,bloquant,85
4,DIPLOME_MANQUANT,bloquant,1
5,ECOLE_GENERIQUE,informatif,12079
6,NIVEAU_COMPLETE_PAR_LIBELLE,informatif,8571
7,ECOLE_MANQUANTE,informatif,2281
8,NIVEAU_SAISI_INTERPRETE,informatif,1871
9,LIBELLE_A_FOURCHETTE,informatif,259


In [42]:
# Contrôle : statut final des lignes
(df["statut"].value_counts().to_frame("nb_lignes")
   .assign(**{"% lignes": lambda d: (d["nb_lignes"] / len(df) * 100).round(1)}))

,nb_lignes,% lignes
statut,,
OK,19232,92.3
A ARBITRER - quasi-doublon,911,4.4
A CONFIRMER - école,414,2.0
A VERIFIER - niveau saisi ≠ libellé,191,0.9
A COMPLETER - niveau,85,0.4
A COMPLETER - diplôme,1,0.0


In [43]:
# Aperçu de la base après l'étape 7
df.head(10)

,matricule,diplome,niveau,specialite,ecole,discipline,ligne_source,diplome_norm,specialite_norm,discipline_norm,niveau_normalise,ctrl_niveau_saisi_imprecis,niveau_theorique,type_ligne,ctrl_diplome_manquant,ctrl_libelle_ambigu,ctrl_diplome_non_classe,ctrl_incoherence_niveau,ctrl_niveau_interprete,ctrl_niveau_complete,niveau_fiabilise,ctrl_niveau_arrondi,ctrl_niveau_manquant,rang_niveau,ecole_nom,ecole_pays,ecole_nature,ecole_confiance,ctrl_ecole_manquante,ctrl_ecole_generique,ctrl_ecole_a_confirmer,categorie_diplome,ctrl_categorie_non_trouvee,ctrl_quasi_doublon,flags,statut
0,MATRICULE000001,SECONDAIRE,SECONDAIRE,Autres,LYCEE,NaN,2,SECONDAIRE,None,None,SECONDAIRE,False,SECONDAIRE,DIPLOME,False,False,False,False,False,False,SECONDAIRE,False,False,0.0,LYCEE (NON PRECISE),MAROC,AUTRE,SUR,False,True,False,AUTRES,False,False,ECOLE_GENERIQUE,OK
1,MATRICULE000002,BREVET TECHNICIEN SPECIALISE,BAC+2,Autres,AUTRES,NaN,3,BREVET TECHNICIEN SPECIALISE,None,None,BAC+2,False,BAC+2,DIPLOME,False,False,False,False,False,False,BAC+2,False,False,3.0,None,,NON_PRECISE,SUR,True,False,False,AUTRES,False,False,ECOLE_MANQUANTE,OK
2,MATRICULE000002,BACCALAUREAT,BAC,Autres,LYCEE,SCIENCE,4,BACCALAUREAT,None,SCIENCE,BAC,False,BAC,DIPLOME,False,False,False,False,False,False,BAC,False,False,1.0,LYCEE (NON PRECISE),MAROC,AUTRE,SUR,False,True,False,AUTRES,False,False,ECOLE_GENERIQUE,OK
3,MATRICULE000002,CERTIFICAT D'APTITUDE BANCAIRE,BAC,DOMAINE BANCAIRE,C.F.P.B.,NaN,5,CERTIFICAT D'APTITUDE BANCAIRE,DOMAINE BANCAIRE,None,BAC,False,BAC,DIPLOME,False,False,False,False,False,False,BAC,False,False,1.0,CFPB - CENTRE DE FORMATION DE LA PROFESSION BANCAIRE,MAROC,AUTRE,SUR,False,False,False,AUTRES,False,False,,OK
4,MATRICULE000003,DIPLOME DES ETUDES UNIVERSITAIRES GNLE.,BAC+2,DROIT,UNIVMARCDR,DROIT,6,DIPLOME DES ETUDES UNIVERSITAIRES GNLE.,DROIT,DROIT,BAC+2,False,BAC+2,DIPLOME,False,False,False,False,False,False,BAC+2,False,False,3.0,UNIVERSITE MAROCAINE (NON PRECISEE),MAROC,UNIVERSITE,SUR,False,True,False,DIPLOME UNIVERSITAIRE,False,False,ECOLE_GENERIQUE,OK
6,MATRICULE000003,BACCALAUREAT,BAC,LETTRES MODERNES,LYCEE,LETTRES MODERNES,8,BACCALAUREAT,LETTRES MODERNES,LETTRES MODERNES,BAC,False,BAC,DIPLOME,False,False,False,False,False,False,BAC,False,False,1.0,LYCEE (NON PRECISE),MAROC,AUTRE,SUR,False,True,False,AUTRES,False,False,ECOLE_GENERIQUE,OK
7,MATRICULE000004,SECONDAIRE,SECONDAIRE,Autres,LYCEE,NaN,9,SECONDAIRE,None,None,SECONDAIRE,False,SECONDAIRE,DIPLOME,False,False,False,False,False,False,SECONDAIRE,False,False,0.0,LYCEE (NON PRECISE),MAROC,AUTRE,SUR,False,True,False,AUTRES,False,False,ECOLE_GENERIQUE,OK
8,MATRICULE000005,INSTITUT DES TECHNIQUES DE BANQUE,BAC+4,DOMAINE BANCAIRE,C.F.P.B.,NaN,10,INSTITUT DES TECHNIQUES DE BANQUE,DOMAINE BANCAIRE,None,BAC+4,False,BAC+4,DIPLOME,False,False,False,False,False,False,BAC+4,False,False,5.0,CFPB - CENTRE DE FORMATION DE LA PROFESSION BANCAIRE,MAROC,AUTRE,SUR,False,False,False,AUTRES,False,False,,OK
9,MATRICULE000005,BREVET BANCAIRE,BAC+2,DOMAINE BANCAIRE,C.F.P.B.,NaN,11,BREVET BANCAIRE,DOMAINE BANCAIRE,None,BAC+2,False,BAC+2,DIPLOME,False,False,False,False,False,False,BAC+2,False,False,3.0,CFPB - CENTRE DE FORMATION DE LA PROFESSION BANCAIRE,MAROC,AUTRE,SUR,False,False,False,AUTRES,False,False,,OK
10,MATRICULE000005,DIPLOME DES ETUDES UNIVERSITAIRES GNLE.,BAC+2,GESTION DES ENTREPRISES,UNIVMARCDR,GESTION,12,DIPLOME DES ETUDES UNIVERSITAIRES GNLE.,GESTION DES ENTREPRISES,GESTION,BAC+2,False,BAC+2,DIPLOME,False,False,False,False,False,False,BAC+2,False,False,3.0,UNIVERSITE MAROCAINE (NON PRECISEE),MAROC,UNIVERSITE,SUR,False,True,False,DIPLOME UNIVERSITAIRE,False,False,ECOLE_GENERIQUE,OK


---
## Étape 8 : Vue salarié

**Objectif** : une ligne par salarié, avec son **diplôme le plus élevé**, son niveau et sa catégorie. C'est la table utilisable pour du reporting ou une analyse de rémunération.

**Choix**

- le diplôme le plus élevé est choisi sur le **rang du niveau fiabilisé** (l'expertise comptable compte comme un BAC+5) ;
- les **certifications sont exclues** du calcul (elles n'ont pas de niveau) mais comptées à part ;
- un salarié sans aucun diplôme de niveau connu est marqué `NON DETERMINE`.

In [44]:
# Diplôme le plus élevé = rang du niveau fiabilisé le plus haut ; les certifications n'ont pas de niveau, elles sont comptées à part
dipl = df[df["type_ligne"].eq("DIPLOME")].sort_values(["matricule", "rang_niveau"], ascending=[True, False])
top = dipl.drop_duplicates("matricule").set_index("matricule")

index_mat = sorted(df["matricule"].dropna().unique())
vue = pd.DataFrame({
    "niveau_max": top["niveau_fiabilise"],
    "categorie_diplome_max": top["categorie_diplome"],   # remplace l'ancien type d'école (public / privé)
    "diplome_max": top["diplome"],
    "ecole_max": top["ecole_nom"],
    "nb_diplomes": df[df["type_ligne"].eq("DIPLOME")].groupby("matricule").size(),
    "nb_certifications": df[df["type_ligne"].eq("CERTIFICATION")].groupby("matricule").size(),
    "nb_lignes_a_traiter": df[df["statut"].ne("OK")].groupby("matricule").size(),
}).reindex(index_mat)
vue["niveau_max"] = vue["niveau_max"].fillna("NON DETERMINE")   # aucun diplôme de niveau connu
for c in ["nb_diplomes", "nb_certifications", "nb_lignes_a_traiter"]:
    vue[c] = vue[c].fillna(0).astype(int)
vue = vue.rename_axis("matricule").reset_index()

print(f"{len(vue):,} salariés".replace(",", " "))
vue.head(10)

8 715 salariés


,matricule,niveau_max,categorie_diplome_max,diplome_max,ecole_max,nb_diplomes,nb_certifications,nb_lignes_a_traiter
0,MATRICULE000001,SECONDAIRE,AUTRES,SECONDAIRE,LYCEE (NON PRECISE),1,0,0
1,MATRICULE000002,BAC+2,AUTRES,BREVET TECHNICIEN SPECIALISE,None,3,0,0
2,MATRICULE000003,BAC+2,DIPLOME UNIVERSITAIRE,DIPLOME DES ETUDES UNIVERSITAIRES GNLE.,UNIVERSITE MAROCAINE (NON PRECISEE),2,0,0
3,MATRICULE000004,SECONDAIRE,AUTRES,SECONDAIRE,LYCEE (NON PRECISE),1,0,0
4,MATRICULE000005,BAC+4,AUTRES,INSTITUT DES TECHNIQUES DE BANQUE,CFPB - CENTRE DE FORMATION DE LA PROFESSION BANCAIRE,3,0,0
5,MATRICULE000006,BAC+2,AUTRES,BREVET TECHNICIEN SPECIALISE,None,1,0,0
6,MATRICULE000007,BAC+4,AUTRES,INSTITUT DES TECHNIQUES DE BANQUE,CFPB - CENTRE DE FORMATION DE LA PROFESSION BANCAIRE,2,0,0
7,MATRICULE000008,BAC+4,AUTRES,INSTITUT DES TECHNIQUES DE BANQUE,CFPB - CENTRE DE FORMATION DE LA PROFESSION BANCAIRE,3,0,0
8,MATRICULE000009,BAC+2,AUTRES,BREVET BANCAIRE,CFPB - CENTRE DE FORMATION DE LA PROFESSION BANCAIRE,1,0,0
9,MATRICULE000010,BAC,AUTRES,CERTIFICAT D'APTITUDE BANCAIRE,CFPB - CENTRE DE FORMATION DE LA PROFESSION BANCAIRE,1,0,0


In [45]:
# Contrôle : répartition des salariés par niveau le plus élevé
(vue["niveau_max"].value_counts()
   .reindex(list(NIVEAUX)[::-1] + ["NON DETERMINE"]).dropna().astype(int)
   .to_frame("nb_salaries"))

,nb_salaries
niveau_max,
BAC+7,18
BAC+6,5
EXPERTISE COMPTABLE,5
BAC+5,4183
BAC+4,774
BAC+3,2046
BAC+2,1521
BAC,135
SECONDAIRE,22


---
## Étape 9 : Export et synthèse

Le fichier de sortie contient :

| Onglet | Contenu |
|---|---|
| `Synthese` | indicateurs avant / après |
| `BDD_Diplomes` | la base fiabilisée complète : le diplôme saisi, son niveau (écrit, théorique, retenu) et sa catégorie ; pour l'école, la spécialité et la discipline, la saisie d'origine (`_source`) et sa dernière version ; puis le statut et les contrôles |
| `A_traiter` | uniquement les lignes dont le statut n'est pas OK, filtrables par motif |
| `Vue_salarie` | une ligne par salarié, diplôme le plus élevé |
| `Doublons_supprimes` | les lignes retirées, avec leur numéro de ligne d'origine |
| `Controles` | nombre de lignes par contrôle |
| `REF_diplomes`, `REF_ecoles` | les correspondances source → fiabilisé, avec effectifs, pour relecture |

In [46]:
# Diplôme et niveau : le libellé saisi (diplome), le niveau saisi (niveau_ecrit), ce que le libellé dit
# du niveau (niveau_theorique) et le niveau retenu (niveau). Pourquoi pas diplome_norm, niveau_normalise
# ni rang_niveau : ce sont des étapes de calcul (majuscules, écriture standard, rang), utiles dans le notebook
# mais pas pour relire le fichier.
# Pour les autres informations : la saisie d'origine (_source) et sa dernière version.
# Spécialité et discipline n'ont pas d'autre traitement que le nettoyage de l'étape 1 (_norm).
bdd = df[[
    "matricule", "type_ligne",
    "diplome", "categorie_diplome",
    "niveau", "niveau_theorique", "niveau_fiabilise",
    "specialite", "specialite_norm", "discipline", "discipline_norm",
    "ecole", "ecole_nom", "ecole_pays", "ecole_nature", "ecole_confiance",
    "statut", "flags", "ligne_source",
]].rename(columns={
    "niveau": "niveau_ecrit", "niveau_fiabilise": "niveau",
    "specialite": "specialite_source", "discipline": "discipline_source", "ecole": "ecole_source",
})

# Référentiels : chaque correspondance source -> fiabilisé avec son effectif, pour relecture
ref_diplomes = (bdd.groupby(["diplome", "categorie_diplome", "niveau"], dropna=False)
                   .size().rename("nb_lignes").reset_index().sort_values("nb_lignes", ascending=False))
ref_ecoles = (bdd.groupby(["ecole_source", "ecole_nom", "ecole_pays", "ecole_nature", "ecole_confiance"], dropna=False)
                 .size().rename("nb_lignes").reset_index().sort_values("nb_lignes", ascending=False))

synthese = pd.DataFrame([
    ("Lignes", len(brut), len(bdd)),
    ("Salariés", brut["Matricule fictif"].nunique(), vue.shape[0]),
    ("Libellés de diplôme distincts", brut["Nom de diplôme ou certification"].nunique(), None),
    ("Écritures de niveau distinctes", brut["Niveau du diplôme"].nunique(), bdd["niveau"].nunique()),
    ("Lignes diplôme sans niveau", int(brut["Niveau du diplôme"].isna().sum()), int(df["ctrl_niveau_manquant"].sum())),
    ("Écoles distinctes", brut["Ecole Diplome ou Organisme Certification"].nunique(), bdd["ecole_nom"].nunique()),
    ("Lignes OK", None, int(bdd["statut"].eq("OK").sum())),
    ("Lignes à traiter", None, int(bdd["statut"].ne("OK").sum())),
], columns=["indicateur", "avant", "apres"])
synthese

,indicateur,avant,apres
0,Lignes,27242.0,20834.0
1,Salariés,8715.0,8715.0
2,Libellés de diplôme distincts,214.0,NaN
3,Écritures de niveau distinctes,25.0,10.0
4,Lignes diplôme sans niveau,11794.0,85.0
5,Écoles distinctes,334.0,260.0
6,Lignes OK,NaN,19232.0
7,Lignes à traiter,NaN,1602.0


In [47]:
onglets = {
    "Synthese": synthese,
    "BDD_Diplomes": bdd,
    "A_traiter": bdd[bdd["statut"].ne("OK")].sort_values(["statut", "matricule"]),
    "Vue_salarie": vue,
    "Doublons_supprimes": doublons[["ligne_source"] + COLS],
    "Controles": tab_controles,
    "REF_diplomes": ref_diplomes,
    "REF_ecoles": ref_ecoles,
}

# Mise en forme : en-tête figé, filtres, largeur de colonne plafonnée à 55 pour rester lisible
with pd.ExcelWriter(FICHIER_SORTIE, engine="openpyxl") as xw:
    for nom, d in onglets.items():
        d.to_excel(xw, sheet_name=nom, index=False)
        ws = xw.sheets[nom]
        ws.freeze_panes = "A2"
        ws.auto_filter.ref = ws.dimensions
        for col in ws.columns:
            largeur = max((len(str(c.value)) for c in col[:300] if c.value is not None), default=8)
            ws.column_dimensions[col[0].column_letter].width = min(max(10, largeur + 2), 55)

print(f"Fichier écrit : {FICHIER_SORTIE}")
for nom, d in onglets.items():
    print(f"  {nom:<20} {len(d):>6} lignes")

Fichier écrit : BDD_Diplomes_FINAL.xlsx
  Synthese                  8 lignes
  BDD_Diplomes          20834 lignes
  A_traiter              1602 lignes
  Vue_salarie            8715 lignes
  Doublons_supprimes     6408 lignes
  Controles                14 lignes
  REF_diplomes            358 lignes
  REF_ecoles              330 lignes


---
## Pour la prochaine campagne

**Relancer** : déposer la nouvelle extraction dans le dossier du notebook, remplacer `FICHIER_SOURCE`, puis *Run > Run All Cells*. Vérifier les niveaux écrits inconnus (étape 2), la liste « écoles hors référentiel » (étape 5) et les libellés non classés (étape 3) : ce sont les seules valeurs nouvelles qui demandent une décision.

**Où modifier quoi**

| Besoin | Cellule à modifier |
|---|---|
| Ajouter un vide déguisé | `PLACEHOLDERS` (étape 1) |
| Traiter un nouveau niveau écrit | `NIVEAUX_ECRITS` (étape 2) |
| Ajouter un niveau standard | `NIVEAUX` (étape 2) |
| Changer l'arrondi des niveaux « et plus » et « ou » | `ARRONDI_NIVEAU` (étape 4) |
| Changer ce qu'un libellé dit de la catégorie | `INDICES_CATEGORIE` (étape 3) |
| Classer un nouveau libellé de diplôme | `REF_DIPLOMES_MANUEL` (cas isolé) ou `REGLES_DIPLOME` (règle générale) — étape 3 |
| Faire primer le niveau saisi sur le libellé | `PRIORITE_NIVEAU = "SAISIE"` (étape 4) |
| Décoder une nouvelle école, confirmer un code ou changer la nature d'une école | `REF_ECOLES` (étape 5) |

**Points ouverts** (onglet `A_traiter`) : les quasi-doublons, les écoles `A_CONFIRMER`, les incohérences de niveau et les niveaux manquants nécessitent un retour au dossier salarié.